# Budget app: read, run, break, fix (slices 1–2)

You know almost no TypeScript. That is fine. This notebook starts with pictures and plain words, then small functions, then one real request, then labs where you change things and watch what happens.

**How to use it**

- The app's source code is **not modified**. Every code cell either imports the real file or copies a small private helper with a note saying where it came from.
- Run the cells **top to bottom** the first time. Some cells reuse names from earlier cells.
- When you see **Predict first**, write your answer on paper before running the next cell.
- Exercises have three parts: **run it**, **break it on purpose**, and **find and fix the bug**.
- Solutions are at the bottom, inside collapsible `<details>` blocks.
- Every section has a time estimate. If you are tired, stop after one section. The notebook will still be here.

**What you need:** the repo at `~/Documents/Projects/budget`, the setup in the next cell, and patience. You do **not** need to know React, TypeScript, or databases before starting.

## Setup: Deno + Jupyter on Pop!_OS (⏱️ ~10 min)

Pop!_OS is Ubuntu-based, so these commands work in its terminal. They install a Python environment for Jupyter and Deno, a TypeScript runtime that can talk to Jupyter.

```bash
# 1. Python + JupyterLab
sudo apt update
sudo apt install -y python3-venv curl
python3 -m venv ~/.venvs/budget-guide
~/.venvs/budget-guide/bin/pip install --upgrade pip jupyterlab

# 2. Deno
curl -fsSL https://deno.land/install.sh | sh
export PATH="$HOME/.deno/bin:$PATH"

# 3. The Deno kernel for Jupyter
deno jupyter --install

# 4. Let Deno read the app's imports that omit ".ts"
python3 - <<'PY'
import json, pathlib
p = pathlib.Path.home() / ".local/share/jupyter/kernels/deno/kernel.json"
spec = json.loads(p.read_text())
if "--sloppy-imports" not in spec["argv"]:
    spec["argv"].insert(2, "--sloppy-imports")
p.write_text(json.dumps(spec, indent=2) + "\n")
print("patched", p)
PY

# 5. Start Jupyter from the learn folder
cd ~/Documents/Projects/budget/learn
~/.venvs/budget-guide/bin/jupyter lab slice-2-guide.ipynb
```

In Jupyter, open the notebook and choose **Kernel → Change kernel → Deno**. If "Deno" appears in the kernel list, setup worked.

**Why step 4 is needed:** files inside `src/` import each other like `from './calendar'` without writing `.ts`. Node and Vite understand that; Deno asks for the extension unless we turn on "sloppy imports". The patch adds that flag to the kernel command.

**Verified:** this notebook was executed end-to-end with `jupyter nbconvert --execute` using Deno 2.9.8, TypeScript 6.0.3, and JupyterLab 4.6.5. If your versions are newer, the commands are the same.

The file `learn/deno.json` is the import map. It tells Deno where to find the npm packages the app uses:

```json
{
  "imports": {
    "dexie": "npm:dexie@4.4.7",
    "fake-indexeddb/auto": "npm:fake-indexeddb@6.2.7/auto",
    "dexie-react-hooks": "npm:dexie-react-hooks@4.4.0",
    "react": "npm:react@19.3.1",
    "react-dom/server": "npm:react-dom@19.3.1/server",
    "react-dom/client": "npm:react-dom@19.3.1/client",
    "jsdom": "npm:jsdom@30.1.3"
  },
  "compilerOptions": {
    "jsx": "react-jsx",
    "jsxImportSource": "react"
  }
}
```

In [ ]:
console.log("Deno:", Deno.version.deno);
console.log("TypeScript:", Deno.version.typescript);
console.log("V8:", Deno.version.v8);
console.log("Working folder:", Deno.cwd());
console.log("If the folder does not end in /learn, restart Jupyter from the learn folder.");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

In [ ]:
import "fake-indexeddb/auto";
import { BudgetDatabase } from "../src/data/db.ts";
import {
  createLedger,
  LedgerError,
  UNCATEGORIZED_CATEGORY_ID,
  UNCATEGORIZED_LABEL,
} from "../src/data/ledger.ts";
import { STARTER_CATEGORIES } from "../src/data/seed.ts";
import {
  parseAmount,
  formatAmount,
  formatAmountInput,
} from "../src/domain/money.ts";
import {
  isValidMonthKey,
  isValidLocalDate,
  todayLocalDate,
  monthOf,
  addMonths,
  formatMonth,
  MIN_MONTH,
  MAX_MONTH,
} from "../src/domain/calendar.ts";
import { parseHash, routeToHash } from "../src/app/router.ts";
import { expenseFieldErrors } from "../src/app/expenseForm.tsx";
import { ACCOUNT_TYPE_LABELS } from "../src/app/labels.ts";

console.log("App imports ready.");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

## 1. Project map: how the pieces fit (⏱️ ~12 min)

### The one-sentence version

A phone screen asks the user for an amount, a category, an account, and a date; the app checks those values, writes one row into the phone's local database, and then reads the month's rows back to draw a total and a category list.

### The pieces, in plain words

- **`main.tsx` is the front door.** It creates the database object, wraps it in a `ledger` object, and calls `boot`.
- **`boot` is the startup sequence.** It seeds the database, draws the app, asks the browser to keep the data, and registers the offline worker.
- **`App.tsx` is the switchboard.** It looks at the URL hash (`#/...`) and decides which screen to show.
- **Screens are the five pages.** Home shows a month. Add expense is the form. Accounts is first-run setup and account management. Category entries shows the rows behind one category. Entry editor corrects or removes one row.
- **`ledger.ts` is the only door to data.** Screens never talk to the database directly. They call `ledger.addExpense`, `ledger.monthSummary`, and so on.
- **`db.ts` is the database shape.** It tells Dexie (a small library) which tables and indexes exist.
- **`domain/` is the math and vocabulary.** `money.ts` converts text like `"12.34"` to `1234` cents and back. `calendar.ts` checks dates and moves between months. `types.ts` names the shapes of accounts, categories, entries, and settings.
- **`seed.ts` is a list of nine starter categories.** It has no functions.
- **`router.ts` reads and writes the URL hash.** The hash is the part after `#`, so the app can change screens without a server.
- **`pwa.ts` makes the app installable and offline.** A service worker caches the app shell.
- **`expenseForm.tsx` is the shared form pieces.** The add form and the edit form use the same amount/category/account/date controls.

### Text diagram

```text
┌────────────────────────────── PHONE ──────────────────────────────┐
│ index.html                                                        │
│   └─ main.tsx                                                     │
│       ├─ new BudgetDatabase()          data/db.ts                 │
│       ├─ createLedger(db)              data/ledger.ts             │
│       └─ boot(ledger, #root)                                      │
│             ├─ ledger.initialize()  → settings + 9 categories     │
│             └─ <App ledger={ledger} />                            │
│                   ├─ router.ts  (which screen?)                   │
│                   └─ screens/                                     │
│                        ├─ HomeScreen                              │
│                        ├─ AddExpenseScreen                        │
│                        ├─ AccountsScreen                          │
│                        ├─ CategoryEntriesScreen                   │
│                        └─ EntryEditorScreen                       │
│                             │                                     │
│                             ▼                                     │
│                        ledger.ts  (the only door)                 │
│                        ├─ validates inputs                        │
│                        ├─ db.ts → Dexie → IndexedDB               │
│                        ├─ seed.ts (starter categories)            │
│                        └─ domain/money.ts, calendar.ts, types.ts  │
└───────────────────────────────────────────────────────────────────┘
```

The arrows only go one way: screens → ledger → database/domain. The domain never imports a screen, and a screen never imports Dexie directly.

### Where slices 1–2 sit in the 8-slice plan

| # | Slice | What the user can do | Status |
|---|---|---|---|
| 1 | Log spending and see this month by category | Add accounts, log an expense, see the month's total and category breakdown, move between months, install and use offline. | **Done** |
| 2 | Inspect and correct entries | Tap a category to see its entries; fix or remove a mistaken entry. | **Done** |
| 3 | Keep a backup | Export the ledger to a file and restore it; see the last backup date. | Backlog |
| 4 | Manage categories | Add, rename, and archive categories. | Backlog |
| 5 | Compare with last month | See last month's category totals beside this month's. | Backlog |
| 6 | Track account balances | Enter opening balances and see what each account holds or owes. | Backlog |
| 7 | Record income | Record money coming in, categorized, without inflating spending. | Backlog |
| 8 | Record transfers | Move money between your own accounts, including paying a credit card. | Backlog |

**Slices 1–2 are the whole current app.** Every file in `src/` is part of them. Slice 1 built the loop "log an expense → see the month". Slice 2 made that loop auditable: you can drill into a category and correct the row that made the number wrong.

### What later slices will build on

- **Slice 3 (backup)** will read and write the same four tables (`accounts`, `categories`, `entries`, `settings`) through `db.ts`. It will not need a new ledger seam.
- **Slice 4 (categories)** will extend `categories` and `listCategories`/`addExpense` validation. The category table already exists, and `Category` already has an `archived` flag.
- **Slice 5 (last month)** will call `monthSummary` twice and use `addMonths` to get the previous month. The read model is already there.
- **Slice 6 (balances)** will use `accounts` and entries. `EntryKind` already includes `'opening'`, but nothing creates it yet.
- **Slice 7 (income)** will use `EntryKind` `'income'`. `monthSummary` already ignores non-expense entries, so income will not inflate spending.
- **Slice 8 (transfers)** will use `EntryKind` `'transfer'` and the optional `toAccountId` field. `Entry` already has that field.

## 2. Every file, in plain words (⏱️ ~15 min)

### The file table

| File | What it is | Inputs | Outputs | Connects to |
|---|---|---|---|---|
| `src/domain/types.ts` | Names for the shapes of data. No code runs. | none | TypeScript types | imported everywhere |
| `src/domain/money.ts` | Money text ↔ integer cents, and display formatting. | strings, numbers, currency, exponent, locale | number, string, `null`, or a `RangeError` | ledger, screens |
| `src/domain/calendar.ts` | Date and month checking, moving, and labeling. | date/month strings, `Date`, delta, locale | boolean, string, or a `RangeError` | router, ledger, screens |
| `src/data/db.ts` | Dexie database class and schema v1. | database name | typed tables | main, ledger, tests |
| `src/data/seed.ts` | Nine starter categories as data. | none | a readonly array | `ledger.initialize` |
| `src/data/ledger.ts` | The app's whole API: validation, transactions, reads. | a database + operation inputs | promises of accounts/entries/summaries | screens, main, tests |
| `src/app/router.ts` | Hash-route parsing, encoding, navigation, and a React hook. | hash text, a `Route`, a `Date` | `Route`, hash text, or `void` | App, screens |
| `src/app/labels.ts` | Display names for account types. | `AccountType` | string | expense form, screens |
| `src/app/expenseForm.tsx` | Shared expense fields and error mapping. | React props | React elements | AddExpense, EntryEditor |
| `src/app/pwa.ts` | Service-worker registration and persistent-storage request. | browser APIs | `void` | main |
| `src/app/BootError.tsx` | The screen shown when the database cannot open. | an error | React element | main |
| `src/app/App.tsx` | Shell, first-run gate, and route switch. | `ledger` | one screen | router, screens |
| `src/app/screens/HomeScreen.tsx` | Month total, category breakdown, month buttons. | `ledger`, `month` | React element | ledger, money, calendar, router |
| `src/app/screens/AddExpenseScreen.tsx` | Guards + the add form. | `ledger` | React element | ledger, money, calendar, expenseForm, router |
| `src/app/screens/AccountsScreen.tsx` | Account list, add form, inline rename. | `ledger` | React element | ledger, labels, router |
| `src/app/screens/CategoryEntriesScreen.tsx` | The entries behind one category and month. | `ledger`, `month`, `categoryId` | React element | ledger, money, calendar, labels, router |
| `src/app/screens/EntryEditorScreen.tsx` | Load, correct, and remove one entry. | `ledger`, `entryId` | React element | ledger, money, calendar, expenseForm, router |
| `src/main.tsx` | Constructs the database/ledger and boots the app. | `ledger`, root element | `void` | db, ledger, App, BootError, pwa |
| `src/test/setup.ts` | Test-only setup: fake IndexedDB and DOM matchers. | none | side effects | Vitest |
| `src/test/harness.test.ts` | A tiny test proving the test setup works. | none | test result | Vitest |
| `src/**/*.test.ts(x)` | The real test files for money, calendar, ledger, router, and screens. | none | pass/fail | Vitest, fake-indexeddb, React Testing Library |
| `src/vite-env.d.ts` | A one-line TypeScript reference for Vite's extra types. | none | type information | TypeScript |
| `src/styles.css` | Plain global CSS. | none | styles | all screens |
| `index.html` | The page with `<div id="root">` and the module script. | none | DOM | `main.tsx` |
| `vite.config.ts` | Vite + PWA build configuration. | none | build config | build |
| `tsconfig.json` | TypeScript compiler settings. | none | compiler config | build/tests |
| `package.json` | Scripts and dependencies. | none | npm metadata | everything |

### File-by-file details

**`src/domain/types.ts`** is a dictionary of shapes, not a program. `Account` says an account has an id, a name, a type, and a creation time. `Entry` says one movement has a kind, a date, an amount in minor units, an account, and optional category/destination fields. `Settings` has one row with id `'app'`, a currency, an exponent, and an optional last-used account. Because it is types only, it disappears when the app runs. Every other module imports from it, which keeps names consistent.

**`src/domain/money.ts`** has three public functions and one private guard. `parseAmount` takes what the user typed and returns an integer number of minor units, or `null` if the text is not a simple money value. `formatAmountInput` does the reverse for an edit field, keeping trailing zeros. `formatAmount` adds a currency symbol for display. `requireExponent` makes sure the currency's decimal count is a whole number from 0 to 6. All three public functions call it first, so a bad exponent fails loudly instead of producing a strange number.

**`src/domain/calendar.ts`** has nine functions. Three are private helpers: `pad2`, `isLeapYear`, and `daysInMonth`. The public ones check a month key, check a local date, read today's local date, take the month out of a date, add or subtract months, and format a month name. The functions throw `RangeError` for bad input instead of returning a wrong answer. This module has no database or React code, so it is easy to test and reuse.

**`src/data/db.ts`** defines `BudgetDatabase`, a class that extends Dexie. Its constructor registers schema version 1 with four tables: `accounts`, `categories`, `entries`, and `settings`. The string `'id, createdAt'` means "primary key is `id`, and also index `createdAt`". The string `'id, date'` on entries means entries can be queried by date. This file does not contain business rules; it only describes storage.

**`src/data/seed.ts`** exports `STARTER_CATEGORIES`, a readonly array of nine category objects. Each has a stable id like `cat_groceries`, a display name, and `archived: false`. There are no functions here. `initialize` copies these rows into the database only if they are missing.

**`src/data/ledger.ts`** is the biggest and most important file. It exports input/output interfaces, the `Ledger` interface, the `LedgerError` class, two uncategorized constants, and the `createLedger` factory. `createLedger` returns an object with twelve async operations: initialize, list accounts, add/rename account, list categories, add/update/delete expense, get one entry, list category entries, month summary, and get settings. Each operation validates its input, uses a Dexie transaction when it writes, and returns plain data. The UI only sees this object, never Dexie.

**`src/app/router.ts`** turns URL hash text into a `Route` object and back. `parseHash` is pure: give it `"#/category?m=2026-11&c=cat_fun"` and it returns `{ name: 'category', month: '2026-11', categoryId: 'cat_fun' }`. `routeToHash` does the reverse. `navigate` writes the hash to the browser. `useHashRoute` is a React hook that subscribes to the browser's `hashchange` event and returns the current route. This is how the app changes screens without a server.

**`src/app/labels.ts`** is a small lookup table from account type to display text: `checking` → `"Checking"`, `savings` → `"Savings"`, `credit` → `"Credit card"`. It exists so the same words appear in the account list and the expense form.

**`src/app/expenseForm.tsx`** contains three exports. `expenseFieldErrors` maps a ledger error code to the field that should show it. `ChoiceGroup` renders a group of chip buttons with a legend, a selected state, and an optional error. `ExpenseFields` renders the four shared inputs (amount, category, account, date) and calls the parent when something changes. The add form and the edit form both use these pieces, so the two forms cannot drift apart.

**`src/app/pwa.ts`** has two small functions. `requestPersistentStorage` asks the browser to keep the app's data and ignores refusal. `registerPwa` registers the service worker generated by `vite-plugin-pwa` with `immediate: true`. Both are best-effort browser calls; neither touches the ledger.

**`src/app/BootError.tsx`** is a fallback screen. If `boot` cannot initialize the database, it renders a heading, a plain message, and a Reload button. It deliberately does not show the raw error, because a beginner user cannot act on it.

**`src/app/App.tsx`** is the top-level component. It reads the current route, loads the account list with a live query, and redirects to the accounts screen when there are no accounts. Otherwise it switches on the route name and renders one of the five screens. It is the only place that knows the full screen list.

**`src/app/screens/HomeScreen.tsx`** loads `monthSummary(month)` and `getSettings()`. It formats the total and each category row, renders previous/next month buttons, and turns each category row into a button that opens the category entries screen. When the month has no expenses, it shows an empty state with an "Add your first expense" button.

**`src/app/screens/AddExpenseScreen.tsx`** loads accounts, categories, and settings. If there are no accounts, it shows a guard message and a link to Accounts. If there are no categories, it shows another guard. Otherwise it renders the private `AddExpenseForm`, which holds the form state, validates the fields, calls `ledger.addExpense`, and navigates home on success.

**`src/app/screens/AccountsScreen.tsx`** lists accounts and has two forms: one to add an account, one to rename an account inline. It keeps separate state for the add form, the rename editor, and errors. It uses refs to block double submits. Once at least one account exists, it shows a "Start logging" button that goes to the current month's home.

**`src/app/screens/CategoryEntriesScreen.tsx`** loads `categoryEntries(month, categoryId)` and settings. It formats the total and each row, and it labels account names. If two different accounts share a name, it adds the account type to avoid confusion. Clicking a row opens the entry editor for that row.

**`src/app/screens/EntryEditorScreen.tsx`** loads one entry, all accounts, all categories, and settings. If the entry is missing or is not an expense, it shows `EntryNotFound`. Otherwise it renders `EntryForm`, which prefills the fields, validates changes, calls `ledger.updateExpense`, and offers a two-step remove action.

**`src/main.tsx`** has `boot` and the top-level startup code. `boot` creates the React root, calls `ledger.initialize()`, renders either `BootError` or `App`, then requests persistent storage and registers the PWA. The last lines find `#root`, create `BudgetDatabase`, wrap it in `createLedger`, and call `boot`.

**`src/test/setup.ts`** imports `fake-indexeddb/auto` so tests have an in-memory IndexedDB, and imports jest-dom matchers. **`src/test/harness.test.ts`** is a small sanity test. These are not part of the running app; they support the test suite.

**The `*.test.ts(x)` files** are the project's safety net. `money.test.ts` and `calendar.test.ts` test pure functions. `ledger.test.ts` runs `createLedger` against a real Dexie database on `fake-indexeddb`, one unique database per test. `router.test.ts` tests hash parsing and encoding. `App.test.tsx` and the screen tests render real components with a real ledger and drive them with React Testing Library. They are not imported by the app; run them with `npm test`. Reading a test is often the fastest way to learn a function's contract.

**`src/vite-env.d.ts`** is a one-line reference that tells TypeScript about Vite's special types, such as `import.meta.env` and the `virtual:pwa-register` module. It has no runtime behavior.

**`src/styles.css`** is a plain global stylesheet. **`index.html`** is the page shell with `<div id="root">`. **`vite.config.ts`** configures Vite and the PWA plugin. **`tsconfig.json`** tells TypeScript how strict to be. **`package.json`** lists the scripts (`dev`, `build`, `test`, `typecheck`) and the dependencies. You do not need to edit any of these to understand slices 1–2.

## 3. Function walkthroughs (⏱️ 4 blocks, ~15–20 min each)

Every walkthrough has the same shape:

1. Five plain sentences saying what the function does.
2. A **Before / After** example. "Before" is the input or the state before the call; "After" is the return value or the state after.

"Private" means the function is not exported. You cannot import it, but it still runs inside its module. In the labs, we copy private helpers into the notebook when we need to call them directly.

The order is easy to hard: tiny calendar helpers → date checks → money → ledger → router/form helpers → screens and startup.

### 3a. Tiny calendar helpers (⏱️ ~15 min)

#### `pad2(value)`

1. `pad2` takes a whole number and turns it into text with at least two characters.
2. If the number is already two digits, it returns it unchanged.
3. If the number is one digit, it puts a `"0"` in front.
4. It is used when building dates like `"2026-01-05"`, so months and days always have two digits.
5. It is private to `calendar.ts`, so the app calls it indirectly through date functions.

**Before:** `pad2(7)`  
**After:** `"07"`  
**Before:** `pad2(12)`  
**After:** `"12"`

#### `isLeapYear(year)`

1. It answers whether a year has 366 days instead of 365.
2. A year is a leap year if it divides evenly by 4.
3. The exception is century years: 1900 is not a leap year.
4. The exception to the exception is years divisible by 400: 2000 is a leap year.
5. `daysInMonth` uses it to decide whether February has 28 or 29 days.

**Before:** `isLeapYear(2024)`  
**After:** `true`  
**Before:** `isLeapYear(1900)`  
**After:** `false`  
**Before:** `isLeapYear(2000)`  
**After:** `true`

#### `daysInMonth(year, month)`

1. It returns how many days a given month has in a given year.
2. February is special because it depends on `isLeapYear`.
3. April, June, September, and November always have 30 days.
4. Every other month has 31 days.
5. `isValidLocalDate` uses it to reject impossible dates like `2023-02-29`.

**Before:** `daysInMonth(2024, 2)`  
**After:** `29`  
**Before:** `daysInMonth(2026, 11)`  
**After:** `30`

#### `isValidMonthKey(value)`

1. It checks whether text looks like a real month key in `YYYY-MM` form.
2. The year must be four digits from 0001 to 9999.
3. The month must be two digits from 01 to 12.
4. It does not accept `2026-1`, `2026-13`, or `0000-01`.
5. The router uses it to ignore a bad `?m=` value and fall back to the current month.

**Before:** `isValidMonthKey("2026-11")`  
**After:** `true`  
**Before:** `isValidMonthKey("2026-13")`  
**After:** `false`

#### `isValidLocalDate(value)`

1. It checks whether text is a real calendar date in `YYYY-MM-DD` form.
2. It reads the year, month, and day as numbers.
3. It checks the ranges and then asks `daysInMonth` whether the day exists.
4. That is why `2024-02-29` passes but `2023-02-29` fails.
5. The add and edit forms use it before saving an expense.

**Before:** `isValidLocalDate("2024-02-29")`  
**After:** `true`  
**Before:** `isValidLocalDate("2023-02-29")`  
**After:** `false`

#### `todayLocalDate(now)`

1. It turns a `Date` object into the local calendar date as `YYYY-MM-DD` text.
2. It uses local year, month, and day, not UTC, so near midnight the date matches the phone's clock.
3. It adds a leading zero to the month and day when needed.
4. It can take a `Date` for testing; with no argument it uses the current time.
5. The add form uses it to prefill today's date.

**Before:** `todayLocalDate(new Date(2026, 10, 4))`  
**After:** `"2026-11-04"`

#### `monthOf(date)`

1. It takes a valid local date and returns its first seven characters.
2. Those seven characters are the month key, such as `"2026-11"`.
3. It rejects an invalid date with a `RangeError` instead of guessing.
4. The app uses it to send the user to the month that contains a saved entry.
5. It is a small bridge between date text and month text.

**Before:** `monthOf("2026-11-04")`  
**After:** `"2026-11"`  
**Before:** `monthOf("2026-13-01")`  
**After:** throws `RangeError`

#### `addMonths(month, delta)`

1. It moves a month key forward or backward by a whole number of months.
2. It converts the year and month to a total month count, adds the delta, and converts back.
3. That handles year boundaries: November plus two months is January of the next year.
4. It throws if the month is malformed, the delta is not a whole number, or the result leaves years 1..9999.
5. The home screen's previous and next buttons call it with `-1` and `1`.

**Before:** `addMonths("2026-11", 2)`  
**After:** `"2027-01"`  
**Before:** `addMonths("2026-01", -1)`  
**After:** `"2025-12"`

#### `formatMonth(month, locale)`

1. It turns a month key into a human label like `"November 2026"`.
2. It validates the key first.
3. It builds a `Date` at noon on the first day of that month to avoid time-zone edge cases.
4. It asks `Intl.DateTimeFormat` to write the month name and year in the chosen locale.
5. The home and category screens use it in their headings.

**Before:** `formatMonth("2026-11", "en-CA")`  
**After:** `"November 2026"`

### 3b. Money functions (⏱️ ~15 min)

#### `requireExponent(exponent)`

1. It checks that a currency exponent is a whole number from 0 to 6.
2. The exponent says how many decimal places a currency uses; CAD uses 2.
3. If the value is wrong, it throws a `RangeError` with a clear message.
4. If the value is right, it returns nothing and lets the caller continue.
5. All three public money functions call it first, so bad exponents never reach the math.

**Before:** `requireExponent(2)`  
**After:** no output  
**Before:** `requireExponent(7)`  
**After:** throws `RangeError`

#### `parseAmount(input, exponent)`

1. It turns money text from a form into an integer number of minor units.
2. For CAD, minor units are cents, so `"12.34"` becomes `1234`.
3. It accepts a dot or a comma as the decimal separator, and it trims spaces.
4. It rejects more decimal places than the currency allows, negative signs, and text that is not a simple number.
5. It returns `null` for bad input so the form can show an error instead of crashing.

**Before:** `parseAmount("12,34", 2)`  
**After:** `1234`  
**Before:** `parseAmount("12.345", 2)`  
**After:** `null`  
**Before:** `parseAmount("1,234.56", 2)`  
**After:** `null`

#### `formatAmountInput(amountMinor, exponent)`

1. It does the opposite of `parseAmount` for an edit form: integer minor units back to editable text.
2. It keeps exactly the right number of decimal places, including trailing zeros.
3. It puts a minus sign in front for negative values.
4. It throws if the amount is not a safe integer or the exponent is invalid.
5. The entry editor uses it to prefill the amount field.

**Before:** `formatAmountInput(1234, 2)`  
**After:** `"12.34"`  
**Before:** `formatAmountInput(5, 2)`  
**After:** `"0.05"`  
**Before:** `formatAmountInput(-4500, 2)`  
**After:** `"-45.00"`

#### `formatAmount(amountMinor, currency, exponent, locale)`

1. It turns integer minor units into a display string with a currency symbol.
2. It divides by `10 ** exponent` to get the major-unit number.
3. It uses `Intl.NumberFormat` so the symbol and grouping match the locale.
4. It forces the currency's exact number of decimal places.
5. The home, category, and entry screens use it for every visible amount.

**Before:** `formatAmount(1234, "CAD", 2, "en-CA")`  
**After:** `"$12.34"`  
**Before:** `formatAmount(1200, "CAD", 0, "en-CA")`  
**After:** `"$1,200"`

> **Locale note:** the app calls `formatAmount` without a locale, so the phone's locale decides the symbol. In this notebook we pass `"en-CA"` when we want a stable `$12.34`. A US locale may show `CA$12.34` for Canadian dollars.

### 3c. Ledger functions (⏱️ ~20 min)

The ledger is a factory: `createLedger(db)` returns an object. The functions below are the methods of that object, plus two private helpers used inside it. All methods are `async`, so callers write `await`.

#### `requireAccountName(name)`

1. It trims spaces from an account name.
2. It rejects an empty name or a name longer than 60 characters.
3. It throws a `LedgerError` with code `invalid-name` when the name is bad.
4. It returns the trimmed name when the name is good.
5. `addAccount` and `renameAccount` both use it, so the rule lives in one place.

**Before:** `requireAccountName("  Visa  ")`  
**After:** `"Visa"`  
**Before:** `requireAccountName("   ")`  
**After:** throws `LedgerError` with code `invalid-name`

#### `createLedger(db)`

1. It is a factory: it takes a database and returns an object full of operations.
2. The object is the app's only door to stored data.
3. Each operation is an `async` function, so callers use `await`.
4. Inside, the operations validate inputs, run Dexie transactions, and return plain objects.
5. Screens receive this object as a prop and never touch Dexie directly.

**Before:** `createLedger(db)`  
**After:** a `Ledger` object; after `await ledger.initialize()` the database is seeded.

#### `requireExpenseFields(input)` (private inside `createLedger`)

1. It checks the four fields every expense needs: amount, date, account, and category.
2. The amount must be a positive safe integer.
3. The date must be a real local date.
4. The account and category must already exist in the database.
5. It throws a specific `LedgerError` code for the first problem it finds.

**Before:** valid input  
**After:** no output  
**Before:** `amountMinor: 0`  
**After:** throws `LedgerError` with code `invalid-amount`

#### `initialize()`

1. It runs once at startup to make the database usable.
2. It creates the single settings row if it is missing.
3. It adds each of the nine starter categories if it is missing.
4. It does not overwrite existing rows, so running it twice is safe.
5. It wraps the work in one transaction so the database never ends up half-seeded.

**Before:** empty database  
**After:** settings row + 9 categories  
**Before:** already-seeded database  
**After:** unchanged (idempotent)

#### `listAccounts()`

1. It reads every account from the database.
2. It sorts them by `createdAt`, oldest first.
3. If two accounts were created at the same instant, it sorts by `id` to keep the order stable.
4. It returns a plain array of `Account` objects.
5. The accounts screen and the add-expense form use it to build their lists.

**Before:** database with 3 accounts  
**After:** sorted array, oldest first

#### `addAccount({ name, type })`

1. It validates and trims the name.
2. It checks that the type is one of `checking`, `savings`, or `credit`.
3. It builds a new account with a random UUID and the current time.
4. It writes the account inside a transaction and returns it.
5. It does not create an opening balance entry; that is a later slice.

**Before:** `{ name: "Visa", type: "credit" }`  
**After:** an `Account` object; the accounts table has one more row

#### `renameAccount(id, name)`

1. It validates and trims the new name.
2. It updates the account row by id.
3. If no row matched the id, it throws `unknown-account`.
4. It does not change entries; they still point at the same account id.
5. The accounts screen calls it from the inline rename form.

**Before:** `renameAccount(id, "  Chequing  ")`  
**After:** no return value; the stored name is `"Chequing"`

#### `listCategories()`

1. It reads every category from the database.
2. It sorts them by name, then by id for ties.
3. It returns a plain array of `Category` objects.
4. The add and edit forms use it to build the category choices.
5. It includes archived categories because slice 2 has no category management yet.

**Before:** seeded database  
**After:** 9 categories, sorted alphabetically

#### `addExpense(input)`

1. It runs inside one read-write transaction covering accounts, categories, entries, and settings.
2. It calls `requireExpenseFields` to validate the input.
3. It builds an expense entry with a random UUID and `kind: "expense"`.
4. It writes the entry and remembers the chosen account in settings.
5. It returns the new entry so the form can navigate to its month.

**Before:** `{ amountMinor: 1234, categoryId: "cat_groceries", accountId, date: "2026-11-04" }`  
**After:** an `Entry`; entries +1, and `settings.lastAccountId` is the account

#### `monthSummary(month)`

1. It validates the month key.
2. It reads every entry whose date starts with that month.
3. It ignores non-expense entries and adds up amounts by category.
4. It looks up category names and gives missing categories the label `"Uncategorized"`.
5. It sorts rows largest first, then by name, and returns the total plus rows.

**Before:** entries 1234 groceries + 500 fun in `2026-11`  
**After:** `{ month: "2026-11", totalMinor: 1734, rows: [Groceries 1234, Fun 500] }`

#### `updateExpense(id, input)`

1. It runs in a transaction and loads the existing entry by id.
2. It throws `unknown-entry` if the entry is gone.
3. It throws `unsupported-entry-kind` if the entry is not an expense.
4. It validates the new fields with the same rules as `addExpense`.
5. It updates the row in place and returns the updated entry.

**Before:** amount 1234  
**After:** amount 1500; same id, still one row

#### `deleteExpense(id)`

1. It runs in a transaction and loads the entry by id.
2. It throws `unknown-entry` if the entry is gone.
3. It throws `unsupported-entry-kind` if the entry is not an expense.
4. It deletes the row and returns the entry that was removed.
5. The entry editor uses the returned date and category to choose where to go next.

**Before:** one entry  
**After:** no entry; entries -1

#### `getEntry(id)`

1. It asks the database for one entry by id.
2. It returns the entry if it exists.
3. It returns `null` if it does not exist.
4. It does not throw for a missing entry, because the editor needs to show a friendly "not found" screen.
5. The entry editor calls it with the id from the URL.

**Before:** `getEntry("missing")`  
**After:** `null`  
**Before:** `getEntry(validId)`  
**After:** the `Entry`

#### `categoryEntries(month, categoryId)`

1. It validates the month key.
2. It reads all entries for that month and keeps only expenses in the requested category.
3. It sorts them newest first, then by id for ties.
4. It looks up the account name and type for each row, using `"Unknown account"` when needed.
5. It returns the category name, total, and rows for the category screen.

**Before:** two groceries entries in `2026-11`  
**After:** `{ categoryName: "Groceries", totalMinor: ..., rows: [newest first] }`

#### `getSettings()`

1. It reads the single settings row with id `"app"`.
2. It throws a plain `Error` if the row is missing, because that means the app was not initialized.
3. It returns the settings object with currency and exponent.
4. The screens use it to format money correctly.
5. It is a read-only operation.

**Before:** initialized database  
**After:** `{ id: "app", currency: "CAD", currencyExponent: 2 }`

### 3d. Router, shared form, and screen helpers (⏱️ ~20 min)

#### `parseHash(hash, today)`

1. It turns the part of the URL after `#` into a `Route` object.
2. It splits the path from the query string.
3. It handles `/add`, `/accounts`, `/category`, `/entry/:id`, and the default home route.
4. It validates the month and falls back to today's month when `?m=` is missing or bad.
5. It is a pure function, so tests can call it without a browser.

**Before:** `parseHash("#/category?m=2026-11&c=cat_fun")`  
**After:** `{ name: "category", month: "2026-11", categoryId: "cat_fun" }`

#### `routeToHash(route)`

1. It does the reverse of `parseHash`: it turns a `Route` object into hash text.
2. It uses `encodeURIComponent` for ids so special characters are safe.
3. It always includes the month for home and category routes.
4. It returns a string beginning with `#`.
5. `navigate` uses it to change the URL.

**Before:** `{ name: "home", month: "2026-11" }`  
**After:** `"#/?m=2026-11"`

#### `navigate(route, options)`

1. It converts a route to a hash and writes it to `window.location`.
2. With `replace: true`, it replaces the current history entry instead of adding one.
3. Replacing is used after saving or deleting so the Back button does not return to a stale form.
4. It returns nothing; the browser's `hashchange` event triggers the screen update.
5. It only works in a browser-like environment.

**Before:** `navigate({ name: "add" })`  
**After:** URL hash is `#/add`

#### `useHashRoute()`

1. It is a React hook that returns the current route.
2. It starts from `parseHash(window.location.hash)`.
3. It subscribes to the browser's `hashchange` event.
4. When the hash changes, it parses the new hash and updates React state.
5. It removes the listener when the component unmounts.

**Before:** URL `#/add`  
**After:** `{ name: "add" }`; after a hash change, the new route

#### `expenseFieldErrors(code)`

1. It maps a `LedgerError` code to the form field that should show the message.
2. For example, `invalid-amount` becomes an `amount` error.
3. It returns `null` for codes that are not field-specific.
4. The add and edit forms use it to show errors next to the right input.
5. It keeps error copy in one place.

**Before:** `expenseFieldErrors("invalid-date")`  
**After:** `{ date: "Choose a valid date." }`

#### `ChoiceGroup(props)`

1. It is a small React component that renders a group of chip buttons.
2. It receives a legend, options, the selected id, an optional error, and a click handler.
3. It marks the selected button with `aria-pressed="true"`.
4. It shows the error text below the group when there is one.
5. The expense form uses it for categories and accounts.

**Before:** props with two options  
**After:** an HTML `fieldset` with two buttons

#### `ExpenseFields(props)`

1. It renders the four shared expense inputs: amount, category, account, and date.
2. It receives the current values and change handlers from its parent.
3. It calls `onClearError` when the user edits a field so old errors disappear.
4. It connects labels to inputs with `htmlFor` and `id` for screen readers.
5. Both the add form and the edit form use it, so they behave the same.

**Before:** props with values  
**After:** HTML inputs; after typing, the parent state changes

#### `accountErrorMessage(code)`

1. It maps an account-related `LedgerError` code to a human message.
2. It handles `invalid-name`, `invalid-account-type`, and `unknown-account`.
3. Any other code gets a generic "try again" message.
4. The accounts screen uses it for both add and rename errors.
5. It is private to the accounts screen.

**Before:** `accountErrorMessage("invalid-name")`  
**After:** `"Enter an account name of 60 characters or fewer."`

#### `accountLabel(row, ambiguousNames)`

1. It decides how to label an account in an entry row.
2. If the account name is unique among the rows, it returns just the name.
3. If two different accounts share the name, it adds the account type.
4. If the account type is `null` (missing account), it returns the name unchanged.
5. The category entries screen uses it to avoid confusing duplicate names.

**Before:** `{ accountName: "Visa", accountType: "credit" }` with duplicate `"Visa"`  
**After:** `"Visa · Credit card"`

#### `duplicateAccountNames(rows)`

1. It finds account names that belong to more than one account id.
2. It builds a map from name to a set of ids.
3. It keeps only names whose set has more than one id.
4. It returns a `Set` of those names.
5. `accountLabel` uses that set to decide when to add the account type.

**Before:** rows from two different `"Visa"` accounts  
**After:** `Set { "Visa" }`

### 3e. Screens, App, boot, and PWA (⏱️ ~20 min)

These are React components and startup functions. A component takes **props** (inputs) and returns **JSX** (a description of HTML). You do not need to memorize React to follow the flow.

#### `App({ ledger })`

1. It is the top-level React component after boot.
2. It reads the current route with `useHashRoute`.
3. It loads the account list with a live query.
4. If there are no accounts, it redirects to the accounts screen so first-run setup happens first.
5. Otherwise it switches on the route and renders one screen.

**Before:** ledger with no accounts  
**After:** `AccountsScreen`  
**Before:** ledger with accounts and hash `#/add`  
**After:** `AddExpenseScreen`

#### `HomeScreen({ ledger, month })`

1. It shows one month's spending.
2. It loads the month summary and settings with live queries.
3. It formats the total and each row with `formatAmount`.
4. The previous and next buttons call `addMonths` and `navigate`.
5. Category rows are buttons that open the category entries screen.

**Before:** month `2026-11` with a 1234-cent grocery expense  
**After:** heading "November 2026", total, and a Groceries button

#### `AddExpenseScreen({ ledger })`

1. It loads accounts, categories, and settings.
2. It shows a guard message if there are no accounts or no categories.
3. Otherwise it renders `AddExpenseForm`.
4. It passes the loaded data down so the form does not fetch again.
5. It is the screen the home "Add expense" button opens.

**Before:** no accounts  
**After:** "Add an account before logging an expense."  
**Before:** accounts exist  
**After:** the form

#### `AddExpenseForm({ ledger, accounts, categories, settings })`

1. It holds the form state: amount text, category, account, date, errors, and submitting.
2. It preselects the last-used account if it still exists, otherwise the first account.
3. Its `submit` function parses the amount, validates all fields, and calls `ledger.addExpense`.
4. On success it navigates to the month of the saved entry.
5. On failure it maps the `LedgerError` to field errors.

**Before:** empty form  
**After:** defaults (today's date, first/last account)  
**Before:** valid save  
**After:** new entry and home screen

#### `clearError(field)` (inside AddExpenseForm)

1. It removes one field's error from the error object.
2. It returns the same object if that field has no error, avoiding an unnecessary render.
3. It copies the object before changing it, because React state must be replaced, not mutated.
4. The shared fields call it whenever the user edits.
5. It is a small helper inside the form component.

**Before:** `{ amount: "..." }` and clear `"amount"`  
**After:** `{}`

#### `submit(event)` (inside AddExpenseForm)

1. It stops the browser's normal form submission.
2. It guards against a double tap with `submittingRef`.
3. It parses and validates the fields, filling `nextErrors`.
4. If anything is wrong, it shows the errors and returns without writing.
5. If everything is right, it calls `ledger.addExpense` and navigates on success.

**Before:** invalid amount  
**After:** error message, no database write  
**Before:** valid fields  
**After:** one entry

#### `AccountsScreen({ ledger })`

1. It lists accounts and lets the user add or rename them.
2. It uses live queries so the list updates after a write.
3. It has separate state for the add form and the inline rename form.
4. It guards double submits with refs.
5. It sends the user to the current month's home when they press "Start logging".

**Before:** no accounts  
**After:** hint + add form  
**Before:** one account  
**After:** list row + rename button

#### `addAccount(event)` (inside AccountsScreen)

1. It handles the add-account form submission.
2. It prevents the default page reload and checks the busy ref.
3. It calls `ledger.addAccount` with the current name and type.
4. On success it clears the form.
5. On failure it shows a mapped error message.

**Before:** `{ name: "Visa", type: "credit" }`  
**After:** new row  
**Before:** empty name  
**After:** error message

#### `startRename(id, currentName)`

1. It turns on inline editing for one account.
2. It stores the account id and its current name.
3. It copies the current name into the draft input.
4. It clears any previous rename error.
5. The row then renders an input and Save/Cancel buttons.

**Before:** no editing  
**After:** editing row for that account

#### `cancelRename()`

1. It turns off inline editing.
2. It clears the rename error.
3. It does not write anything to the database.
4. The row goes back to its normal display.
5. It is the safe escape from a rename.

**Before:** editing  
**After:** not editing

#### `saveRename(event)`

1. It handles the rename form submission.
2. It checks that an account is being edited and that no save is already running.
3. It calls `ledger.renameAccount` with the draft name.
4. On success it closes the editor.
5. On failure it shows a mapped error and keeps the editor open.

**Before:** draft `"Chequing"`  
**After:** database updated  
**Before:** blank draft  
**After:** error, editor stays open

#### `CategoryEntriesScreen({ ledger, month, categoryId })`

1. It shows the entries behind one category in one month.
2. It loads them with `ledger.categoryEntries`.
3. It formats the total and each row's amount.
4. It labels duplicate account names with their type.
5. Clicking a row opens the entry editor.

**Before:** category groceries with two entries  
**After:** list newest first, plus total

#### `EntryEditorScreen({ ledger, entryId })`

1. It loads one entry, all accounts, all categories, and settings.
2. It shows a "not found" screen if the entry is missing or is not an expense.
3. Otherwise it renders `EntryForm` with the loaded data.
4. It keys the form by entry id so switching entries resets the form state.
5. It is opened by the `/entry/:id` route.

**Before:** missing id  
**After:** "Entry not found"  
**Before:** valid id  
**After:** the form

#### `EntryForm({ ledger, entry, accounts, categories, settings })`

1. It holds the edit state for one expense.
2. It prefills the amount as editable text and the other fields from the entry.
3. It shows errors if the stored category or account no longer exists.
4. Its `submit` updates the entry and navigates to the category list.
5. Its `confirmRemove` deletes the entry and chooses the next screen.

**Before:** entry 1234  
**After:** amount `"12.34"`  
**Before:** save  
**After:** updated row

#### `clearError(field)` (inside EntryForm)

1. Same idea as the add form's helper.
2. It removes one field error.
3. It keeps the same object when nothing changes.
4. It copies before changing.
5. The shared fields call it on edit.

**Before:** `{ date: "..." }` and clear `"date"`  
**After:** `{}`

#### `submit(event)` (inside EntryForm)

1. It prevents the default form action and guards double submits.
2. It parses and validates the same four fields.
3. If valid, it calls `ledger.updateExpense`.
4. On success it navigates to the category list for the updated entry's month and category.
5. If the entry disappeared, it shows the not-found screen.

**Before:** valid change  
**After:** updated entry  
**Before:** missing entry  
**After:** not-found screen

#### `confirmRemove()` (inside EntryForm)

1. It guards double deletes.
2. It calls `ledger.deleteExpense`.
3. It reads the removed entry's month and category.
4. It checks whether any entries remain in that category and month.
5. It navigates to the category list if some remain, otherwise to home.

**Before:** one entry  
**After:** home  
**Before:** two entries  
**After:** category list

#### `EntryNotFound()`

1. It is a small screen shown when an entry cannot be edited.
2. It explains that the entry no longer exists.
3. It offers a "Go home" button.
4. The button navigates with `replace: true` so the bad URL is not kept in history.
5. It is used by both the loader and the form after a failed save/delete.

**Before:** rendered  
**After:** HTML "Entry not found"

#### `BootError({ error })`

1. It is shown when the app cannot open its local database.
2. It does not show the raw error to the user.
3. It offers a Reload button.
4. Reload re-runs the app from scratch.
5. It is rendered by `boot` instead of `App`.

**Before:** an error  
**After:** the error screen

#### `requestPersistentStorage()`

1. It asks the browser to keep the app's data even when storage is tight.
2. It checks that `navigator.storage.persist` exists first.
3. It calls it and ignores the result.
4. It catches failures so a refusal does not break startup.
5. It is best-effort, not a guarantee.

**Before:** browser with the API  
**After:** a persist request  
**Before:** browser without the API  
**After:** no-op

#### `registerPwa()`

1. It registers the service worker generated by `vite-plugin-pwa`.
2. `immediate: true` means registration starts right away.
3. The service worker precaches the app shell for offline use.
4. It returns nothing.
5. It runs after the app renders.

**Before:** called  
**After:** service-worker registration starts

#### `boot(ledger, rootElement)`

1. It creates the React root for the `#root` element.
2. It calls `ledger.initialize()` before rendering the app.
3. If initialization fails, it renders `BootError` and stops.
4. If it succeeds, it renders `App` inside `StrictMode`.
5. Then it requests persistent storage and registers the PWA.

**Before:** ledger + root element  
**After:** rendered app  
**Before:** initialization error  
**After:** `BootError`

#### Top-level startup (last lines of `main.tsx`)

1. It finds the `#root` element in `index.html`.
2. If found, it creates a `BudgetDatabase`.
3. It wraps the database in `createLedger`.
4. It calls `boot` and ignores the returned promise with `void`.
5. This is the first code that runs when the app loads.

**Before:** page load  
**After:** the app starts

## 4. One request trace: from button click to stored data and back (⏱️ ~10 min)

We will follow one action: **the user taps Save on the Add expense form**. The amount is `12.34`, the category is Groceries, the account is Visa, and the date is `2026-11-04`.

### The path, step by step

1. The user taps **Save**. The button is inside a `<form onSubmit={submit}>`.
2. React calls `submit(event)` in `AddExpenseForm`.
3. `submit` calls `event.preventDefault()` so the browser does not reload the page.
4. `submit` calls `parseAmount("12.34", 2)`, which returns `1234` cents.
5. `submit` checks that a category, an account, and a valid date are present.
6. `submit` calls `ledger.addExpense({ amountMinor: 1234, categoryId: "cat_groceries", accountId, date: "2026-11-04" })`.
7. `addExpense` opens a Dexie transaction over `accounts`, `categories`, `entries`, and `settings`.
8. Inside the transaction, `requireExpenseFields` checks the amount, the date, and that the account and category exist.
9. `addExpense` builds an `Entry` with a random UUID and `kind: "expense"`.
10. `db.entries.add(entry)` writes the row.
11. `addExpense` reads the settings row and writes `lastAccountId` so the next expense preselects the same account.
12. The transaction commits and returns the entry.
13. `submit` calls `navigate({ name: "home", month: monthOf(entry.date) })`.
14. The URL hash becomes `#/?m=2026-11`.
15. The browser fires `hashchange`; `useHashRoute` parses the new hash and updates React state.
16. `App` sees `route.name === "home"` and renders `HomeScreen` with `month = "2026-11"`.
17. `HomeScreen` calls `ledger.monthSummary("2026-11")` through `useLiveQuery`.
18. `monthSummary` reads entries whose date starts with `2026-11`, keeps expenses, sums by category, looks up names, and sorts.
19. React renders the total and the category rows.
20. `useLiveQuery` keeps watching the `entries` table. If another write happens, steps 17–19 run again without a refresh.

### Run the storage half of the trace

This cell uses the real `BudgetDatabase` and `createLedger` from the app, with `fake-indexeddb` as the phone storage. It does not render React; it shows exactly what gets stored.

In [ ]:
var traceDb = new BudgetDatabase("trace-" + crypto.randomUUID());
var traceLedger = createLedger(traceDb);
await traceLedger.initialize();

var traceAccount = await traceLedger.addAccount({ name: "Visa", type: "credit" });
console.log("Account:", traceAccount);

var typedAmount = "12.34";
var amountMinor = parseAmount(typedAmount, 2);
console.log("Step 4: parseAmount(" + JSON.stringify(typedAmount) + ", 2) =", amountMinor);

var traceEntry = await traceLedger.addExpense({
  amountMinor: amountMinor!,
  categoryId: "cat_groceries",
  accountId: traceAccount.id,
  date: "2026-11-04",
});
console.log("Step 9-12: stored entry:", traceEntry);
console.log("Stored rows:", await traceDb.entries.toArray());
console.log("Settings after save:", await traceDb.settings.get("app"));

var traceSummary = await traceLedger.monthSummary("2026-11");
console.log("Step 17-18: monthSummary:", traceSummary);

var traceSettings = await traceLedger.getSettings();
console.log(
  "Step 19: Home would show:",
  formatMonth("2026-11", "en-CA"),
  "— Spent",
  formatAmount(traceSummary.totalMinor, traceSettings.currency, traceSettings.currencyExponent, "en-CA"),
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Run the "back to the screen" half

The next cell simulates what happens after the write: it reads the summary again, then updates the entry and reads the summary again. This is the same loop `useLiveQuery` performs automatically in the browser.

In [ ]:
console.log("Before correction:", (await traceLedger.monthSummary("2026-11")).totalMinor);

var corrected = await traceLedger.updateExpense(traceEntry.id, {
  amountMinor: 1500,
  categoryId: "cat_groceries",
  accountId: traceAccount.id,
  date: "2026-11-05",
});
console.log("Corrected entry:", corrected);
console.log("After correction:", (await traceLedger.monthSummary("2026-11")).totalMinor);

var drillDown = await traceLedger.categoryEntries("2026-11", "cat_groceries");
console.log("Category screen rows:", drillDown.rows);
console.log("Category screen total:", drillDown.totalMinor);

var removed = await traceLedger.deleteExpense(traceEntry.id);
console.log("Removed:", removed.id);
console.log("After removal:", (await traceLedger.monthSummary("2026-11")).totalMinor);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

## 5. Hands-on labs (⏱️ 8 labs, ~10–20 min each)

The labs follow the same pattern:

- **Predict first:** write your answer before running.
- **Run it:** the cell prints real results.
- **Break it on purpose:** the cell calls the function with bad input and shows the error or wrong answer.
- **Find and fix the bug:** a deliberately broken copy fails a check. Fix it, rerun, and watch the check turn green.

The checks print `PASS` or `FAIL`. A `FAIL` is not a crash; it is the exercise working. Solutions are at the bottom.

> **Important:** the notebook uses an in-memory IndexedDB (`fake-indexeddb`). Nothing here touches your real app data.

### Lab setup (⏱️ ~2 min)

Run this once. It imports the real modules and defines three helpers: `check`, `checkRejects`, and `freshLedger`.

In [ ]:
import "fake-indexeddb/auto";
import { BudgetDatabase } from "../src/data/db.ts";
import {
  createLedger,
  LedgerError,
  UNCATEGORIZED_CATEGORY_ID,
  UNCATEGORIZED_LABEL,
} from "../src/data/ledger.ts";
import { STARTER_CATEGORIES } from "../src/data/seed.ts";
import {
  parseAmount,
  formatAmount,
  formatAmountInput,
} from "../src/domain/money.ts";
import {
  isValidMonthKey,
  isValidLocalDate,
  todayLocalDate,
  monthOf,
  addMonths,
  formatMonth,
  MIN_MONTH,
  MAX_MONTH,
} from "../src/domain/calendar.ts";
import { parseHash, routeToHash } from "../src/app/router.ts";
import { expenseFieldErrors } from "../src/app/expenseForm.tsx";
import { ACCOUNT_TYPE_LABELS } from "../src/app/labels.ts";

var checksPassed = 0;
var checksFailed = 0;

function check(name: string, actual: unknown, expected: unknown): void {
  var actualText = JSON.stringify(actual);
  var expectedText = JSON.stringify(expected);
  if (actualText === expectedText) {
    checksPassed += 1;
    console.log("PASS", name);
  } else {
    checksFailed += 1;
    console.log("FAIL", name, "| expected", expectedText, "| got", actualText);
  }
}

function checkThrows(
  name: string,
  fn: () => unknown,
  expectedName: string,
): void {
  try {
    fn();
    checksFailed += 1;
    console.log("FAIL", name, "| did not throw");
  } catch (error) {
    var actualName = error instanceof Error ? error.name : typeof error;
    if (actualName === expectedName) {
      checksPassed += 1;
      console.log("PASS", name);
    } else {
      checksFailed += 1;
      console.log("FAIL", name, "| expected", expectedName, "| got", actualName);
    }
  }
}

async function checkRejects(
  name: string,
  fn: () => Promise<unknown>,
  expectedName: string,
): Promise<void> {
  try {
    await fn();
    checksFailed += 1;
    console.log("FAIL", name, "| did not reject");
  } catch (error) {
    var actualName = error instanceof Error ? error.name : typeof error;
    if (actualName === expectedName) {
      checksPassed += 1;
      console.log("PASS", name);
    } else {
      checksFailed += 1;
      console.log("FAIL", name, "| expected", expectedName, "| got", actualName);
    }
  }
}

async function freshLedger(label: string) {
  var db = new BudgetDatabase("lab-" + label + "-" + crypto.randomUUID());
  var ledger = createLedger(db);
  await ledger.initialize();
  return { db, ledger };
}

function sleep(ms: number): Promise<void> {
  return new Promise((resolve) => setTimeout(resolve, ms));
}

console.log("Lab setup ready.");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.1 — Tiny calendar helpers (⏱️ ~10 min)

**Predict first:** what are `pad2(7)`, `pad2(12)`, `isLeapYear(1900)`, and `daysInMonth(2024, 2)`?

The three functions below are **copied** from `src/domain/calendar.ts` because they are private. The source file is unchanged.

In [ ]:
function pad2(value: number): string {
  return String(value).padStart(2, "0");
}

function isLeapYear(year: number): boolean {
  return year % 4 === 0 && (year % 100 !== 0 || year % 400 === 0);
}

function daysInMonth(year: number, month: number): number {
  switch (month) {
    case 2:
      return isLeapYear(year) ? 29 : 28;
    case 4:
    case 6:
    case 9:
    case 11:
      return 30;
    default:
      return 31;
  }
}

console.log("pad2(7) =", pad2(7));
console.log("pad2(12) =", pad2(12));
console.log("isLeapYear(1900) =", isLeapYear(1900));
console.log("isLeapYear(2000) =", isLeapYear(2000));
console.log("daysInMonth(2024, 2) =", daysInMonth(2024, 2));
console.log("daysInMonth(2026, 11) =", daysInMonth(2026, 11));

check("pad2(7)", pad2(7), "07");
check("pad2(12)", pad2(12), "12");
check("1900 is not leap", isLeapYear(1900), false);
check("2000 is leap", isLeapYear(2000), true);
check("Feb 2024 has 29 days", daysInMonth(2024, 2), 29);
check("Nov 2026 has 30 days", daysInMonth(2026, 11), 30);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** `daysInMonth` is a private helper, so it trusts its caller. If you pass month `13`, it falls through to the default and returns `31`. The public `isValidLocalDate` rejects month `13` before calling it. Run this and notice that there is no error.

In [ ]:
console.log("daysInMonth(2026, 13) =", daysInMonth(2026, 13));
console.log("The public date checker is the guard; this helper assumes a valid month.");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This copy of `daysInMonth` forgot leap years. The check expects 29 for February 2024 and will fail. Fix the function, rerun the cell, and make it pass. The solution is at the bottom.

In [ ]:
function daysInMonthBroken(year: number, month: number): number {
  if (month === 2) {
    return 28;
  }
  if (month === 4 || month === 6 || month === 9 || month === 11) {
    return 30;
  }
  return 31;
}

check("Feb 2024 should have 29 days", daysInMonthBroken(2024, 2), 29);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.2 — Date and month validation (⏱️ ~10 min)

**Predict first:** which of these are valid?

- `isValidMonthKey("2026-11")`
- `isValidMonthKey("2026-13")`
- `isValidLocalDate("2024-02-29")`
- `isValidLocalDate("2023-02-29")`
- `isValidLocalDate("2026-1-01")`

In [ ]:
console.log("2026-11:", isValidMonthKey("2026-11"));
console.log("2026-13:", isValidMonthKey("2026-13"));
console.log("2024-02-29:", isValidLocalDate("2024-02-29"));
console.log("2023-02-29:", isValidLocalDate("2023-02-29"));
console.log("2026-1-01:", isValidLocalDate("2026-1-01"));

check("2026-11 is a month", isValidMonthKey("2026-11"), true);
check("2026-13 is not a month", isValidMonthKey("2026-13"), false);
check("2026-1 is not two digits", isValidMonthKey("2026-1"), false);
check("2024-02-29 is a date", isValidLocalDate("2024-02-29"), true);
check("2023-02-29 is not a date", isValidLocalDate("2023-02-29"), false);
check("0000-01-01 is not a date", isValidLocalDate("0000-01-01"), false);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** The real functions reject impossible values. Notice that they do not throw here; they return `false`. The router and forms use that boolean to choose a fallback or show an error.

In [ ]:
console.log("isValidMonthKey('2026-00') =", isValidMonthKey("2026-00"));
console.log("isValidMonthKey('9999-12') =", isValidMonthKey("9999-12"));
console.log("isValidLocalDate('2026-01-32') =", isValidLocalDate("2026-01-32"));
console.log("isValidLocalDate('2026-01-01 ') =", isValidLocalDate("2026-01-01 "));

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This `isValidMonthKeyBroken` only checks the shape `dddd-dd`, so it accepts `2026-13`. Fix it so the month is `01`–`12` and the year is `0001`–`9999`.

In [ ]:
function isValidMonthKeyBroken(value: string): boolean {
  return /^\d{4}-\d{2}$/.test(value);
}

check("2026-13 should be invalid", isValidMonthKeyBroken("2026-13"), false);
check("2026-11 should be valid", isValidMonthKeyBroken("2026-11"), true);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.3 — Moving and labeling months (⏱️ ~10 min)

**Predict first:** what are `todayLocalDate(new Date(2026, 10, 4))`, `monthOf("2026-11-04")`, `addMonths("2026-11", 2)`, `addMonths("2026-01", -1)`, and `formatMonth("2026-11", "en-CA")`?

In [ ]:
console.log("today:", todayLocalDate(new Date(2026, 10, 4)));
console.log("monthOf:", monthOf("2026-11-04"));
console.log("addMonths +2:", addMonths("2026-11", 2));
console.log("addMonths -1:", addMonths("2026-01", -1));
console.log("formatMonth:", formatMonth("2026-11", "en-CA"));

check("todayLocalDate", todayLocalDate(new Date(2026, 10, 4)), "2026-11-04");
check("monthOf", monthOf("2026-11-04"), "2026-11");
check("addMonths +2", addMonths("2026-11", 2), "2027-01");
check("addMonths -1", addMonths("2026-01", -1), "2025-12");
check("formatMonth", formatMonth("2026-11", "en-CA"), "November 2026");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** `addMonths` rejects a fractional delta, and `monthOf` rejects an invalid date. Run the checks; each should print `PASS` because the function throws the expected error.

In [ ]:
checkThrows("addMonths rejects 1.5", () => addMonths("2026-11", 1.5), "RangeError");
checkThrows("addMonths rejects 2026-13", () => addMonths("2026-13", 1), "RangeError");
checkThrows("monthOf rejects 2026-13-01", () => monthOf("2026-13-01"), "RangeError");
checkThrows("formatMonth rejects 2026-13", () => formatMonth("2026-13", "en-CA"), "RangeError");
checkThrows("addMonths stops at year 1", () => addMonths(MIN_MONTH, -1), "RangeError");
checkThrows("addMonths stops at year 9999", () => addMonths(MAX_MONTH, 1), "RangeError");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This `addMonthsBroken` adds the delta to the month number but never carries into the next year. `2026-11 + 2` becomes `2026-13`, which is not a real month. Fix it using the year/month conversion from the real function.

In [ ]:
function addMonthsBroken(month: string, delta: number): string {
  var year = Number(month.slice(0, 4));
  var monthNumber = Number(month.slice(5, 7)) + delta;
  return `${year}-${String(monthNumber).padStart(2, "0")}`;
}

check("2026-11 + 2 should be 2027-01", addMonthsBroken("2026-11", 2), "2027-01");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.4 — Parsing money (⏱️ ~15 min)

**Predict first:** what does `parseAmount` return for each input at exponent 2?

| Input | Predict |
|---|---|
| `"12"` | ? |
| `"12.3"` | ? |
| `"12,34"` | ? |
| `"0.00"` | ? |
| `"12.345"` | ? |
| `"1,234.56"` | ? |

In [ ]:
console.log('parseAmount("12", 2) =', parseAmount("12", 2));
console.log('parseAmount("12.3", 2) =', parseAmount("12.3", 2));
console.log('parseAmount("12,34", 2) =', parseAmount("12,34", 2));
console.log('parseAmount("0.00", 2) =', parseAmount("0.00", 2));
console.log('parseAmount("12.345", 2) =', parseAmount("12.345", 2));
console.log('parseAmount("1,234.56", 2) =', parseAmount("1,234.56", 2));

check("12 -> 1200", parseAmount("12", 2), 1200);
check("12.3 -> 1230", parseAmount("12.3", 2), 1230);
check("12,34 -> 1234", parseAmount("12,34", 2), 1234);
check("0.00 -> 0", parseAmount("0.00", 2), 0);
check("12.345 -> null", parseAmount("12.345", 2), null);
check("1,234.56 -> null", parseAmount("1,234.56", 2), null);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** `parseAmount` is strict: no negative sign, no thousands separators, no exponent, no trailing dot, and no decimals when the exponent is 0. It also rejects a bad exponent with a `RangeError`.

In [ ]:
check("negative -> null", parseAmount("-1", 2), null);
check("plus sign -> null", parseAmount("+1", 2), null);
check("trailing dot -> null", parseAmount("5.", 2), null);
check("exponent 0 rejects decimals", parseAmount("12.5", 0), null);
check("exponent 0 accepts digits", parseAmount("12", 0), 12);
checkThrows("exponent 7 throws", () => parseAmount("12", 7), "RangeError");
checkThrows("exponent 1.5 throws", () => parseAmount("12", 1.5), "RangeError");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This copy forgets to reject too many decimal places. `parseAmountBroken("12.345", 2)` should be `null`, but it builds a strange number. Add the missing guard.

In [ ]:
function parseAmountBroken(input: string, exponent: number): number | null {
  var match = /^(\d+)(?:[.,](\d+))?$/.exec(input.trim());
  if (!match) {
    return null;
  }
  var wholeText = match[1] ?? "";
  var fractionText = match[2] ?? "";
  return Number(wholeText) * 10 ** exponent + Number(fractionText.padEnd(exponent, "0"));
}

check("12.345 at 2 should be null", parseAmountBroken("12.345", 2), null);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.5 — Formatting money (⏱️ ~15 min)

**Predict first:** what are `formatAmountInput(4500, 2)`, `formatAmountInput(5, 2)`, `formatAmountInput(-4500, 2)`, `formatAmount(1234, "CAD", 2, "en-CA")`, and `formatAmount(1200, "CAD", 0, "en-CA")`?

In [ ]:
console.log("4500 at 2 =", formatAmountInput(4500, 2));
console.log("5 at 2 =", formatAmountInput(5, 2));
console.log("-4500 at 2 =", formatAmountInput(-4500, 2));
console.log("1234 CAD =", formatAmount(1234, "CAD", 2, "en-CA"));
console.log("1200 CAD exponent 0 =", formatAmount(1200, "CAD", 0, "en-CA"));

check("4500 at 2", formatAmountInput(4500, 2), "45.00");
check("5 at 2", formatAmountInput(5, 2), "0.05");
check("-4500 at 2", formatAmountInput(-4500, 2), "-45.00");
check("1234 CAD", formatAmount(1234, "CAD", 2, "en-CA"), "$12.34");
check("1200 CAD exponent 0", formatAmount(1200, "CAD", 0, "en-CA"), "$1,200");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** Both functions reject unsafe amounts and bad exponents. Run the checks; they should pass because the functions throw.

In [ ]:
checkThrows(
  "formatAmountInput rejects unsafe amount",
  () => formatAmountInput(Number.MAX_SAFE_INTEGER + 1, 2),
  "RangeError",
);
checkThrows(
  "formatAmount rejects bad exponent",
  () => formatAmount(100, "CAD", 7, "en-CA"),
  "RangeError",
);
checkThrows(
  "formatAmountInput rejects bad exponent",
  () => formatAmountInput(100, -1),
  "RangeError",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This copy divides and calls `String`, so `4500` at exponent 2 becomes `"45"` and loses the trailing zeros. Fix it so it always shows exactly two decimal places and handles negatives.

In [ ]:
function formatAmountInputBroken(amountMinor: number, exponent: number): string {
  return String(amountMinor / 10 ** exponent);
}

check("4500 at 2 keeps trailing zeros", formatAmountInputBroken(4500, 2), "45.00");
check("5 at 2 keeps the leading zero", formatAmountInputBroken(5, 2), "0.05");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.6 — Ledger basics: initialize, accounts, categories, settings (⏱️ ~15 min)

**Predict first:** after `initialize()`, how many categories and accounts are there? What is the currency? What happens to `"  Visa  "` when it is saved?

In [ ]:
var lab1 = await freshLedger("accounts");
console.log("Categories:", (await lab1.ledger.listCategories()).length);
console.log("Accounts:", (await lab1.ledger.listAccounts()).length);
console.log("Settings:", await lab1.ledger.getSettings());

check("nine starter categories", (await lab1.ledger.listCategories()).length, 9);
check("no accounts at first", (await lab1.ledger.listAccounts()).length, 0);
check("currency is CAD", (await lab1.ledger.getSettings()).currency, "CAD");

var acct1 = await lab1.ledger.addAccount({ name: "  Visa  ", type: "credit" });
console.log("Added account:", acct1);

await lab1.ledger.renameAccount(acct1.id, "  Chequing  ");
console.log("Renamed account:", await lab1.db.accounts.get(acct1.id));

check("name is trimmed on add", acct1.name, "Visa");
check("one account after add", (await lab1.ledger.listAccounts()).length, 1);
check("name is trimmed on rename", (await lab1.db.accounts.get(acct1.id))?.name, "Chequing");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** The ledger rejects an empty name, a bad account type, and a rename of an account that does not exist. Each check should print `PASS` because the promise rejects with a `LedgerError`.

In [ ]:
await checkRejects(
  "empty name is rejected",
  () => lab1.ledger.addAccount({ name: "   ", type: "checking" }),
  "LedgerError",
);
await checkRejects(
  "bad account type is rejected",
  () => lab1.ledger.addAccount({ name: "Cash", type: "cash" as any }),
  "LedgerError",
);
await checkRejects(
  "unknown rename is rejected",
  () => lab1.ledger.renameAccount("missing", "Name"),
  "LedgerError",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This `requireAccountNameBroken` returns the name unchanged, so spaces survive and an empty name is accepted. Fix it with `trim()` and the 60-character limit.

In [ ]:
function requireAccountNameBroken(name: string): string {
  return name;
}

check("trims spaces", requireAccountNameBroken("  Visa  "), "Visa");
checkThrows(
  "rejects empty",
  () => requireAccountNameBroken("   "),
  "LedgerError",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.7 — Adding expenses and reading a month (⏱️ ~15 min)

**Predict first:** after adding 1234 cents of Groceries and 500 cents of Fun in `2026-11`, what is `totalMinor`? Which category is first? What is `settings.lastAccountId`?

In [ ]:
var lab2 = await freshLedger("expenses");
var acct2 = await lab2.ledger.addAccount({ name: "Checking", type: "checking" });

var exp1 = await lab2.ledger.addExpense({
  amountMinor: 1234,
  categoryId: "cat_groceries",
  accountId: acct2.id,
  date: "2026-11-04",
});
var exp2 = await lab2.ledger.addExpense({
  amountMinor: 500,
  categoryId: "cat_fun",
  accountId: acct2.id,
  date: "2026-11-02",
});

var sum2 = await lab2.ledger.monthSummary("2026-11");
console.log("Summary:", sum2);
console.log("Settings:", await lab2.ledger.getSettings());

check("total is 1734", sum2.totalMinor, 1734);
check("largest category first", sum2.rows[0]?.categoryId, "cat_groceries");
check("two rows", sum2.rows.length, 2);
check("last account remembered", (await lab2.ledger.getSettings()).lastAccountId, acct2.id);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** The ledger rejects a zero amount, an impossible date, and an unknown category. Each check should pass.

In [ ]:
await checkRejects(
  "zero amount is rejected",
  () => lab2.ledger.addExpense({
    amountMinor: 0,
    categoryId: "cat_groceries",
    accountId: acct2.id,
    date: "2026-11-04",
  }),
  "LedgerError",
);
await checkRejects(
  "impossible date is rejected",
  () => lab2.ledger.addExpense({
    amountMinor: 100,
    categoryId: "cat_groceries",
    accountId: acct2.id,
    date: "2026-02-30",
  }),
  "LedgerError",
);
await checkRejects(
  "unknown category is rejected",
  () => lab2.ledger.addExpense({
    amountMinor: 100,
    categoryId: "cat_missing",
    accountId: acct2.id,
    date: "2026-11-04",
  }),
  "LedgerError",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This summary adds every entry, including income. Income is money coming in, not spending, so it must not inflate the total. Add the `kind === "expense"` filter.

In [ ]:
async function monthSummaryBroken(db: BudgetDatabase, month: string) {
  var entries = await db.entries.where("date").startsWith(month).toArray();
  var totalMinor = entries.reduce((sum, entry) => sum + entry.amountMinor, 0);
  return { totalMinor };
}

var labBug = await freshLedger("summary-bug");
var acctBug = await labBug.ledger.addAccount({ name: "Card", type: "credit" });
await labBug.ledger.addExpense({
  amountMinor: 1234,
  categoryId: "cat_groceries",
  accountId: acctBug.id,
  date: "2026-11-04",
});
await labBug.db.entries.add({
  id: crypto.randomUUID(),
  kind: "income",
  date: "2026-11-06",
  amountMinor: 10000,
  accountId: acctBug.id,
  categoryId: "cat_other",
});

check(
  "income must not count as spending",
  (await monthSummaryBroken(labBug.db, "2026-11")).totalMinor,
  1234,
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.8 — Inspecting, correcting, and removing entries (⏱️ ~15 min)

**Predict first:** after updating the 1234-cent grocery entry to 1500 and deleting it, what does `monthSummary("2026-11")` return each time? What does `categoryEntries` return before the update?

In [ ]:
var cat2 = await lab2.ledger.categoryEntries("2026-11", "cat_groceries");
console.log("Category entries:", cat2);

check("groceries total", cat2.totalMinor, 1234);
check("one groceries row", cat2.rows.length, 1);
check("row amount", cat2.rows[0]?.amountMinor, 1234);

var updated2 = await lab2.ledger.updateExpense(exp1.id, {
  amountMinor: 1500,
  categoryId: "cat_groceries",
  accountId: acct2.id,
  date: "2026-11-05",
});
console.log("Updated:", updated2);
check("updated amount", updated2.amountMinor, 1500);
check(
  "summary after update",
  (await lab2.ledger.monthSummary("2026-11")).totalMinor,
  2000,
);

var removed2 = await lab2.ledger.deleteExpense(exp1.id);
console.log("Removed:", removed2.id);
check("removed id", removed2.id, exp1.id);
check(
  "summary after delete",
  (await lab2.ledger.monthSummary("2026-11")).totalMinor,
  500,
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** Updating a missing entry and deleting the same entry twice both fail with a `LedgerError`. The checks should pass.

In [ ]:
await checkRejects(
  "update missing entry",
  () => lab2.ledger.updateExpense("missing", {
    amountMinor: 100,
    categoryId: "cat_groceries",
    accountId: acct2.id,
    date: "2026-11-04",
  }),
  "LedgerError",
);
await checkRejects(
  "delete twice",
  () => lab2.ledger.deleteExpense(exp1.id),
  "LedgerError",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This `categoryEntriesBroken` sorts oldest first. The real screen promises newest first, so the most recent purchase is easiest to find. Fix the sort.

In [ ]:
var labBug2 = await freshLedger("entries-bug");
var acctBug2 = await labBug2.ledger.addAccount({ name: "Card", type: "credit" });
await labBug2.ledger.addExpense({
  amountMinor: 100,
  categoryId: "cat_fun",
  accountId: acctBug2.id,
  date: "2026-11-02",
});
await labBug2.ledger.addExpense({
  amountMinor: 200,
  categoryId: "cat_fun",
  accountId: acctBug2.id,
  date: "2026-11-04",
});

async function categoryEntriesBroken(
  db: BudgetDatabase,
  month: string,
  categoryId: string,
) {
  var entries = await db.entries.where("date").startsWith(month).toArray();
  var rows = entries
    .filter((entry) => entry.kind === "expense" && entry.categoryId === categoryId)
    .sort((left, right) => left.date.localeCompare(right.date));
  return {
    rows,
    totalMinor: rows.reduce((sum, row) => sum + row.amountMinor, 0),
  };
}

check(
  "newest entry first",
  (await categoryEntriesBroken(labBug2.db, "2026-11", "cat_fun")).rows[0]?.date,
  "2026-11-04",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.9 — Router parsing (⏱️ ~15 min)

**Predict first:** what does each hash parse to?

- `"#/add"`
- `"#/accounts"`
- `"#/?m=2026-11"`
- `"#/category?m=2026-11&c=cat_fun"`
- `"#/entry/abc"`
- `"#/nonsense"` with today `"2026-11-04"`

In [ ]:
console.log("add:", parseHash("#/add"));
console.log("accounts:", parseHash("#/accounts"));
console.log("home:", parseHash("#/?m=2026-11"));
console.log("category:", parseHash("#/category?m=2026-11&c=cat_fun"));
console.log("entry:", parseHash("#/entry/abc"));
console.log("unknown:", parseHash("#/nonsense", "2026-11-04"));

check("add route", parseHash("#/add"), { name: "add" });
check("accounts route", parseHash("#/accounts"), { name: "accounts" });
check("home route", parseHash("#/?m=2026-11"), { name: "home", month: "2026-11" });
check("category route", parseHash("#/category?m=2026-11&c=cat_fun"), {
  name: "category",
  month: "2026-11",
  categoryId: "cat_fun",
});
check("entry route", parseHash("#/entry/abc"), { name: "entry", entryId: "abc" });
check("unknown route falls back", parseHash("#/nonsense", "2026-11-04"), {
  name: "home",
  month: "2026-11",
});

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** A bad month in the query string is ignored and replaced by today's month; the category route still opens. A broken percent-escape in an entry id falls back home. `routeToHash` encodes special characters in ids.

In [ ]:
check("bad month uses today's month", parseHash("#/category?m=2026-13&c=x", "2026-11-04"), {
  name: "category",
  month: "2026-11",
  categoryId: "x",
});
check("bad escape falls back", parseHash("#/entry/%E0%A4%A", "2026-11-04"), {
  name: "home",
  month: "2026-11",
});
check(
  "routeToHash encodes the id",
  routeToHash({ name: "category", month: "2026-11", categoryId: "cat fun" }),
  "#/category?m=2026-11&c=cat%20fun",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This `parseHashBroken` always returns the home route, so `/add`, `/accounts`, `/category`, and `/entry/...` are lost. Fix it by checking the path before falling back to home.

In [ ]:
function parseHashBroken(hash: string, today: string): any {
  var query = hash.split("?")[1] ?? "";
  var requestedMonth = new URLSearchParams(query).get("m");
  var month =
    requestedMonth !== null && isValidMonthKey(requestedMonth)
      ? requestedMonth
      : monthOf(today);
  return { name: "home", month };
}

check("add route", parseHashBroken("#/add", "2026-11-04"), { name: "add" });
check("category route", parseHashBroken("#/category?m=2026-11&c=cat_fun", "2026-11-04"), {
  name: "category",
  month: "2026-11",
  categoryId: "cat_fun",
});

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.10 — Shared form helpers (⏱️ ~10 min)

**Predict first:** what does `expenseFieldErrors("invalid-amount")` return? What does `expenseFieldErrors("unknown-entry")` return? If two different accounts are both named "Visa", how does `accountLabel` display them?

In [ ]:
function accountErrorMessage(code: string): string {
  switch (code) {
    case "invalid-name":
      return "Enter an account name of 60 characters or fewer.";
    case "invalid-account-type":
      return "Choose a valid account type.";
    case "unknown-account":
      return "That account no longer exists.";
    default:
      return "Could not save the account. Try again.";
  }
}

function duplicateAccountNames(rows: any[]): Set<string> {
  var idsByName = new Map<string, Set<string>>();
  for (var row of rows) {
    if (row.accountType === null) {
      continue;
    }
    var ids = idsByName.get(row.accountName) ?? new Set<string>();
    ids.add(row.accountId);
    idsByName.set(row.accountName, ids);
  }
  return new Set(
    [...idsByName]
      .filter(([, ids]) => ids.size > 1)
      .map(([name]) => name),
  );
}

function accountLabel(row: any, ambiguousNames: Set<string>): string {
  if (row.accountType !== null && ambiguousNames.has(row.accountName)) {
    return `${row.accountName} · ${ACCOUNT_TYPE_LABELS[row.accountType]}`;
  }
  return row.accountName;
}

console.log("invalid-amount:", expenseFieldErrors("invalid-amount"));
console.log("unknown-entry:", expenseFieldErrors("unknown-entry"));
console.log("invalid-name:", accountErrorMessage("invalid-name"));

var dupRows = [
  { accountId: "a", accountName: "Visa", accountType: "credit" },
  { accountId: "b", accountName: "Visa", accountType: "credit" },
  { accountId: "c", accountName: "Bank", accountType: "checking" },
];
console.log("duplicates:", [...duplicateAccountNames(dupRows)]);
console.log("ambiguous:", accountLabel(dupRows[0], duplicateAccountNames(dupRows)));
console.log("unique:", accountLabel(dupRows[2], duplicateAccountNames(dupRows)));

check("invalid amount maps to amount", expenseFieldErrors("invalid-amount"), {
  amount: "Enter an amount greater than zero.",
});
check("unknown code has no field", expenseFieldErrors("unknown-entry"), null);
check(
  "invalid name message",
  accountErrorMessage("invalid-name"),
  "Enter an account name of 60 characters or fewer.",
);
check("duplicate names", [...duplicateAccountNames(dupRows)], ["Visa"]);
check(
  "ambiguous label adds type",
  accountLabel(dupRows[0], duplicateAccountNames(dupRows)),
  "Visa · Credit card",
);
check(
  "unique label stays short",
  accountLabel(dupRows[2], duplicateAccountNames(dupRows)),
  "Bank",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** `accountLabel` leaves a name alone when the account type is `null` (the account was deleted), because there is no type to show. It also leaves a unique name alone.

In [ ]:
check(
  "missing account type stays plain",
  accountLabel({ accountName: "Visa", accountType: null }, new Set(["Visa"])),
  "Visa",
);
check(
  "unknown error gets generic copy",
  accountErrorMessage("something-else"),
  "Could not save the account. Try again.",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** This `accountLabelBroken` always adds the account type, even when the name is unique. Fix it so unique names stay short.

In [ ]:
function accountLabelBroken(row: any, ambiguousNames: Set<string>): string {
  return `${row.accountName} · ${ACCOUNT_TYPE_LABELS[row.accountType]}`;
}

check(
  "unique name should not add type",
  accountLabelBroken(dupRows[2], duplicateAccountNames(dupRows)),
  "Bank",
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.11 — Rendering the real screens (⏱️ ~20 min)

This lab uses `jsdom`, a fake browser DOM, to render the **real** React screens with a **real** ledger. Nothing is mocked. You will see the actual HTML the app produces.

**Predict first:** what heading, total, and category button will `HomeScreen` render for November 2026 with a 1234-cent grocery expense?

In [ ]:
import { JSDOM } from "jsdom";
import React from "react";
import { createRoot } from "react-dom/client";
import { act } from "react";
import { HomeScreen } from "../src/app/screens/HomeScreen.tsx";
import { CategoryEntriesScreen } from "../src/app/screens/CategoryEntriesScreen.tsx";
import { EntryEditorScreen } from "../src/app/screens/EntryEditorScreen.tsx";
import { AddExpenseScreen } from "../src/app/screens/AddExpenseScreen.tsx";
import { AccountsScreen } from "../src/app/screens/AccountsScreen.tsx";
import { useHashRoute } from "../src/app/router.ts";
import { App } from "../src/app/App.tsx";
import { BootError } from "../src/app/BootError.tsx";
import { ChoiceGroup } from "../src/app/expenseForm.tsx";

var uiDom = new JSDOM(
  "<!doctype html><html><body><div id='root'></div></body></html>",
  { url: "https://example.test/#/?m=2026-11" },
);
(globalThis as any).window = uiDom.window;
(globalThis as any).document = uiDom.window.document;
(globalThis as any).navigator = uiDom.window.navigator;
(globalThis as any).HTMLElement = uiDom.window.HTMLElement;
(globalThis as any).Event = uiDom.window.Event;
(globalThis as any).IS_REACT_ACT_ENVIRONMENT = true;

var uiDb = new BudgetDatabase("ui-" + crypto.randomUUID());
var uiLedger = createLedger(uiDb);
await uiLedger.initialize();
var uiAccount = await uiLedger.addAccount({ name: "Visa", type: "credit" });
var uiEntry = await uiLedger.addExpense({
  amountMinor: 1234,
  categoryId: "cat_groceries",
  accountId: uiAccount.id,
  date: "2026-11-04",
});

async function renderScreen(element: any): Promise<string> {
  var host = document.getElementById("root")!;
  host.innerHTML = "";
  var root = createRoot(host);
  await act(async () => {
    root.render(element);
    await sleep(300);
  });
  for (var tries = 0; tries < 20; tries += 1) {
    if (!host.innerHTML.includes("Loading…")) {
      break;
    }
    await act(async () => {
      await sleep(100);
    });
  }
  var html = host.innerHTML;
  await act(async () => {
    root.unmount();
  });
  return html;
}

async function renderInteractive(element: any) {
  var host = document.getElementById("root")!;
  host.innerHTML = "";
  var root = createRoot(host);
  await act(async () => {
    root.render(element);
    await sleep(300);
  });
  for (var tries = 0; tries < 20; tries += 1) {
    if (!host.innerHTML.includes("Loading…")) {
      break;
    }
    await act(async () => {
      await sleep(100);
    });
  }
  return root;
}

var homeHtml = await renderScreen(
  React.createElement(HomeScreen, { ledger: uiLedger, month: "2026-11" }),
);
console.log(homeHtml);

check("home heading", homeHtml.includes("November 2026"), true);
check("home total", homeHtml.includes("12.34"), true);
check("home category", homeHtml.includes("Groceries"), true);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Click the category row.** The real `HomeScreen` calls `navigate`, which writes to `window.location.hash`. This is the same path the phone takes when the user taps a category.

In [ ]:
var clickHost = document.getElementById("root")!;
clickHost.innerHTML = "";
var clickRoot = createRoot(clickHost);
await act(async () => {
  clickRoot.render(
    React.createElement(HomeScreen, { ledger: uiLedger, month: "2026-11" }),
  );
  await sleep(300);
});

for (var waitTries = 0; waitTries < 10; waitTries += 1) {
  if (clickHost.querySelector(".category-list button") !== null) {
    break;
  }
  await act(async () => {
    await sleep(100);
  });
}

console.log("HTML before click:", clickHost.innerHTML);
var categoryButton = clickHost.querySelector(".category-list button") as HTMLButtonElement | null;
if (categoryButton === null) {
  console.log("No category button appeared. Run the previous UI cell first.");
} else {
  console.log("Button text:", categoryButton.textContent);
  await act(async () => {
    categoryButton.click();
    await sleep(50);
  });
  console.log("Hash after click:", window.location.hash);
  check(
    "click opens the category route",
    window.location.hash.startsWith("#/category?m=2026-11&c=cat_groceries"),
    true,
  );
}
await act(async () => {
  clickRoot.unmount();
});

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Render the other four screens.** The checks look for a stable piece of text in each screen. This proves the real components mount with real data.

In [ ]:
var categoryHtml = await renderScreen(
  React.createElement(CategoryEntriesScreen, {
    ledger: uiLedger,
    month: "2026-11",
    categoryId: "cat_groceries",
  }),
);
console.log("Category screen:", categoryHtml.slice(0, 400));
check("category heading", categoryHtml.includes("Groceries · November 2026"), true);
check("category row amount", categoryHtml.includes("12.34"), true);

var editorHtml = await renderScreen(
  React.createElement(EntryEditorScreen, { ledger: uiLedger, entryId: uiEntry.id }),
);
console.log("Editor screen:", editorHtml.slice(0, 400));
check("editor heading", editorHtml.includes("Correct expense"), true);
check("editor prefilled amount", editorHtml.includes("12.34"), true);

var addHtml = await renderScreen(
  React.createElement(AddExpenseScreen, { ledger: uiLedger }),
);
console.log("Add screen:", addHtml.slice(0, 400));
check("add heading", addHtml.includes("Add expense"), true);
check("add save button", addHtml.includes("Save"), true);

var accountsHtml = await renderScreen(
  React.createElement(AccountsScreen, { ledger: uiLedger }),
);
console.log("Accounts screen:", accountsHtml.slice(0, 400));
check("accounts heading", accountsHtml.includes("Accounts"), true);
check("accounts list has Visa", accountsHtml.includes("Visa"), true);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Test the hook, the shell, and the small shared pieces.** `useHashRoute` should read the current hash and react when it changes. `App` should choose the screen. `BootError` and `ChoiceGroup` should render their text and states.

In [ ]:
function RouteProbe() {
  var route = useHashRoute();
  return React.createElement("pre", { id: "route" }, JSON.stringify(route));
}

uiDom.window.location.hash = "#/accounts";
var routeRoot = await renderInteractive(React.createElement(RouteProbe));
var routeHost = document.getElementById("root")!;
console.log("Hook route:", routeHost.innerHTML);
check("useHashRoute reads the hash", routeHost.innerHTML.includes('"name":"accounts"'), true);

await act(async () => {
  uiDom.window.location.hash = "#/add";
  await sleep(200);
});
check("useHashRoute reacts to hashchange", routeHost.innerHTML.includes('"name":"add"'), true);
await act(async () => {
  routeRoot.unmount();
});

uiDom.window.location.hash = "#/add";
var appRoot = await renderInteractive(React.createElement(App, { ledger: uiLedger }));
var appHost = document.getElementById("root")!;
check("App renders the add route", appHost.innerHTML.includes("Add expense"), true);
await act(async () => {
  appRoot.unmount();
});

var bootErrorHtml = await renderScreen(
  React.createElement(BootError, { error: new Error("boom") }),
);
check("BootError explains the problem", bootErrorHtml.includes("open its local data"), true);

var choiceHtml = await renderScreen(
  React.createElement(ChoiceGroup, {
    legend: "Pick one",
    options: [
      { id: "a", label: "A" },
      { id: "b", label: "B" },
    ],
    selectedId: "a",
    error: "bad choice",
    errorId: "choice-error",
    onSelect: () => {},
  }),
);
check("ChoiceGroup marks the selected option", choiceHtml.includes('aria-pressed="true"'), true);
check("ChoiceGroup shows the error", choiceHtml.includes("bad choice"), true);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Break it on purpose.** A month with no expenses shows the empty state instead of a misleading zero total. Render January 2025 and look for the empty message.

In [ ]:
var emptyHtml = await renderScreen(
  React.createElement(HomeScreen, { ledger: uiLedger, month: "2025-01" }),
);
console.log(emptyHtml);
check("empty month message", emptyHtml.includes("No spending yet this month."), true);
check("empty month action", emptyHtml.includes("Add your first expense"), true);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Find and fix the bug.** The real `duplicateAccountNames` only treats a name as ambiguous when two different account ids share it. This broken copy treats every name as ambiguous, so a unique "Bank" row would wrongly show its type. Fix it.

In [ ]:
function duplicateAccountNamesBroken(rows: any[]): Set<string> {
  return new Set(rows.map((row) => row.accountName));
}

check(
  "unique Bank is not ambiguous",
  duplicateAccountNamesBroken(dupRows).has("Bank"),
  false,
);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.12 — Real form actions (⏱️ ~20 min)

The previous lab rendered screens. This lab **uses** them: it types into the real inputs, clicks the real buttons, and checks the database afterward. This is the closest the notebook gets to using the phone app.

**Predict first:** when you type `9.99`, choose Fun, set the date, and tap Save, what lands in the database? What does the URL hash become? What happens when you rename an account and then press Cancel?

In [ ]:
var actionDb = new BudgetDatabase("action-" + crypto.randomUUID());
var actionLedger = createLedger(actionDb);
await actionLedger.initialize();
var actionAccount = await actionLedger.addAccount({ name: "Visa", type: "credit" });

var actionDom = new JSDOM(
  "<!doctype html><html><body><div id='root'></div></body></html>",
  { url: "https://example.test/#/?m=2026-11" },
);
(globalThis as any).window = actionDom.window;
(globalThis as any).document = actionDom.window.document;
(globalThis as any).navigator = actionDom.window.navigator;
(globalThis as any).HTMLElement = actionDom.window.HTMLElement;
(globalThis as any).Event = actionDom.window.Event;
(globalThis as any).IS_REACT_ACT_ENVIRONMENT = true;

var actionHost = actionDom.window.document.getElementById("root")!;
var valueSetter = Object.getOwnPropertyDescriptor(
  actionDom.window.HTMLInputElement.prototype,
  "value",
)!.set!;

function reactPropsOf(element: Element): any {
  var keys = Object.keys(element).filter((candidate) =>
    candidate.startsWith("__reactProps"),
  );
  for (var key of keys) {
    var props = (element as any)[key];
    if (props && typeof props.onChange === "function") {
      return props;
    }
  }
  return keys.length > 0 ? (element as any)[keys[0]] : null;
}

async function typeInto(selector: string, value: string): Promise<void> {
  var input = actionHost.querySelector(selector) as HTMLInputElement;
  var props = reactPropsOf(input);
  await act(async () => {
    if (props && typeof props.onChange === "function") {
      // JSDOM does not always deliver native input events to React's
      // synthetic event system. Calling the same onChange prop is the
      // closest reliable simulation in this notebook.
      props.onChange({ target: { value } });
    } else {
      valueSetter.call(input, value);
      input.dispatchEvent(new actionDom.window.Event("input", { bubbles: true }));
    }
    await sleep(100);
  });
  var afterProps = reactPropsOf(input);
  console.log(
    "Typed",
    selector,
    "=",
    value,
    "| React state:",
    afterProps?.value,
  );
}

async function clickText(text: string): Promise<void> {
  var button = Array.from(actionHost.querySelectorAll("button")).find(
    (candidate) => candidate.textContent === text,
  ) as HTMLButtonElement | undefined;
  if (!button) {
    throw new Error("Button not found: " + text);
  }
  await act(async () => {
    button.click();
    await sleep(150);
  });
}

console.log("Action lab ready.");

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Action 1: add an expense through the real form.** The cell types `9.99`, clicks Fun, sets the date, and clicks Save. Then it reads the database directly.

In [ ]:
var addRoot = await renderInteractive(
  React.createElement(AddExpenseScreen, { ledger: actionLedger }),
);
await typeInto("#expense-amount", "9.99");
console.log(
  "Amount input value:",
  (actionHost.querySelector("#expense-amount") as HTMLInputElement).value,
);

var categoryFieldset = Array.from(actionHost.querySelectorAll("fieldset")).find(
  (fieldset) => fieldset.querySelector("legend")?.textContent === "Category",
)!;
var funButton = Array.from(categoryFieldset.querySelectorAll("button")).find(
  (button) => button.textContent === "Fun",
) as HTMLButtonElement;
await act(async () => {
  funButton.click();
  await sleep(100);
});
console.log("Fun pressed:", funButton.getAttribute("aria-pressed"));

await typeInto("#expense-date", "2026-11-04");
console.log(
  "Date input value:",
  (actionHost.querySelector("#expense-date") as HTMLInputElement).value,
);
await clickText("Save");
console.log(
  "Field errors:",
  Array.from(actionHost.querySelectorAll(".field-error")).map(
    (node) => node.textContent,
  ),
);

var savedEntries = await actionDb.entries.toArray();
console.log("Saved entries:", savedEntries);
check("form saved one entry", savedEntries.length, 1);
check("form parsed 9.99", savedEntries[0]?.amountMinor, 999);
check("form chose Fun", savedEntries[0]?.categoryId, "cat_fun");
check(
  "form navigated home",
  actionDom.window.location.hash.startsWith("#/?m=2026-11"),
  true,
);
await act(async () => {
  addRoot.unmount();
});

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Action 2: add, rename, and cancel on the Accounts screen.** This exercises `addAccount`, `startRename`, `saveRename`, and `cancelRename`.

In [ ]:
var accountsRoot = await renderInteractive(
  React.createElement(AccountsScreen, { ledger: actionLedger }),
);

await typeInto("#account-name", "Test");
var addAccountForm = actionHost.querySelector("#account-name")!.closest("form")!;
await act(async () => {
  (addAccountForm.querySelector("button[type=submit]") as HTMLButtonElement).click();
  await sleep(400);
});
check(
  "account added",
  (await actionLedger.listAccounts()).some((account) => account.name === "Test"),
  true,
);

await clickText("Rename");
await typeInto('input[aria-label="Rename Visa"]', "Visa 2");
var renameForm = actionHost
  .querySelector('input[aria-label="Rename Visa"]')!
  .closest("form")!;
await act(async () => {
  (renameForm.querySelector("button[type=submit]") as HTMLButtonElement).click();
  await sleep(400);
});
check(
  "account renamed",
  (await actionLedger.listAccounts()).some((account) => account.name === "Visa 2"),
  true,
);

await clickText("Rename");
await clickText("Cancel");
check(
  "cancel kept the name",
  (await actionLedger.listAccounts()).some((account) => account.name === "Visa 2"),
  true,
);
await act(async () => {
  accountsRoot.unmount();
});

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

**Action 3: correct and remove entries through the real editor.** This exercises `updateExpense`, `confirmRemove`, and `EntryNotFound`.

In [ ]:
var editEntry = await actionLedger.addExpense({
  amountMinor: 1234,
  categoryId: "cat_groceries",
  accountId: actionAccount.id,
  date: "2026-11-04",
});
var deleteEntry = await actionLedger.addExpense({
  amountMinor: 500,
  categoryId: "cat_health",
  accountId: actionAccount.id,
  date: "2026-11-05",
});

var editRoot = await renderInteractive(
  React.createElement(EntryEditorScreen, {
    ledger: actionLedger,
    entryId: editEntry.id,
  }),
);
await typeInto("#entry-amount", "15.00");
await clickText("Save changes");
check(
  "entry updated",
  (await actionLedger.getEntry(editEntry.id))?.amountMinor,
  1500,
);
check(
  "editor navigated to the category",
  actionDom.window.location.hash.includes("/category?"),
  true,
);
await act(async () => {
  editRoot.unmount();
});

var deleteRoot = await renderInteractive(
  React.createElement(EntryEditorScreen, {
    ledger: actionLedger,
    entryId: deleteEntry.id,
  }),
);
await clickText("Remove entry");
check(
  "confirmation names the entry",
  actionHost.innerHTML.includes("This cannot be undone"),
  true,
);
await clickText("Remove");
check("entry removed", await actionLedger.getEntry(deleteEntry.id), null);
check(
  "remove navigated home when the category was empty",
  actionDom.window.location.hash.startsWith("#/?m=2026-11"),
  true,
);
await act(async () => {
  deleteRoot.unmount();
});

var missingRoot = await renderInteractive(
  React.createElement(EntryEditorScreen, {
    ledger: actionLedger,
    entryId: "missing",
  }),
);
check("missing entry screen", actionHost.innerHTML.includes("Entry not found"), true);
await act(async () => {
  missingRoot.unmount();
});

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

### Lab 5.13 — The full request trace with a failure (⏱️ ~10 min)

**Predict first:** what happens if the settings row is deleted and you try to add an expense? Does the entry get written anyway?

The ledger uses a transaction, so the entry write and the settings update succeed or fail together. This lab proves it.

In [ ]:
var trace2Db = new BudgetDatabase("trace2-" + crypto.randomUUID());
var trace2Ledger = createLedger(trace2Db);
await trace2Ledger.initialize();
var trace2Account = await trace2Ledger.addAccount({ name: "Visa", type: "credit" });

var beforeCount = await trace2Db.entries.count();
await trace2Db.settings.delete("app");

await checkRejects(
  "add fails without settings",
  () => trace2Ledger.addExpense({
    amountMinor: 1234,
    categoryId: "cat_groceries",
    accountId: trace2Account.id,
    date: "2026-11-04",
  }),
  "Error",
);

var afterCount = await trace2Db.entries.count();
console.log("Entries before:", beforeCount, "after:", afterCount);
check("transaction rolled back the entry", afterCount, beforeCount);

await trace2Ledger.initialize();
var recovered = await trace2Ledger.addExpense({
  amountMinor: 1234,
  categoryId: "cat_groceries",
  accountId: trace2Account.id,
  date: "2026-11-04",
});
console.log("After initialize, add works:", recovered.id);
check("recovered entry exists", (await trace2Ledger.getEntry(recovered.id))?.id, recovered.id);

// Give the kernel a moment to flush the last printed line.
await new Promise((resolve) => setTimeout(resolve, 50));

## 6. Glossary and quiz (⏱️ ~15 min + ~5 min quiz)

### Glossary

Every technical term used in this notebook, in plain words. Terms are grouped so you can find them by topic.

#### TypeScript and code words

- **TypeScript** — JavaScript with optional type labels. The labels help editors catch mistakes; the browser still runs JavaScript.
- **Type** — a description of what kind of value something is, such as `string` or `number`.
- **Interface** — a named list of fields a value must have, such as `Account`.
- **Union type** — a value that can be one of several types, such as `'checking' | 'savings' | 'credit'`.
- **Optional property** — a field that may be missing. In `categoryId?: string`, the `?` means "may be absent".
- **Literal type** — a type that allows only one exact value, such as `'app'`.
- **Function** — a named block of code that takes inputs and returns an output.
- **Parameter** — the name a function uses for an input inside its body.
- **Argument** — the actual value you pass when you call a function.
- **Return value** — what a function gives back.
- **Pure function** — a function whose output depends only on its inputs and which changes nothing outside itself.
- **Side effect** — anything a function does beyond returning a value, such as writing to a database.
- **Module** — one file of code that exports names for other files.
- **Import / export** — how one module uses a name from another.
- **`async` / `await`** — syntax for waiting on work that takes time, such as a database read.
- **Promise** — an object that represents a future result. `await` waits for it.
- **Callback** — a function passed to another function to be called later.
- **Arrow function** — a short function written `(x) => x + 1`.
- **Array** — an ordered list, written `[1, 2, 3]`.
- **Object** — a collection of named fields, written `{ name: "Visa" }`.
- **`Map`** — a collection of key → value pairs with fast lookup.
- **`Set`** — a collection of unique values.
- **Regular expression (regex)** — a pattern for matching text, such as `/^\d{4}-\d{2}$/`.
- **Integer** — a whole number with no fraction.
- **Safe integer** — an integer JavaScript can represent exactly; `Number.isSafeInteger` checks it.
- **`null`** — a deliberate "no value".
- **`undefined`** — "this was never set".
- **`throw`** — stop the current function and report an error.
- **`try` / `catch`** — run code that might throw, and handle the error if it does.
- **Error class** — a named kind of error, such as `RangeError` or `LedgerError`.
- **Spread syntax (`...`)** — copy the fields of an object or the items of an array into a new one.
- **Optional chaining (`?.`)** — read a field only if the value exists; otherwise return `undefined`.
- **Nullish coalescing (`??`)** — use the right side only when the left side is `null` or `undefined`.
- **`as` assertion** — tell TypeScript "treat this value as this type". It does not change the runtime value.
- **`unknown`** — a type that says "we do not know yet"; you must check before using it.
- **`readonly`** — a promise that a value will not be changed.
- **`keyof`** — a type made from the field names of another type.

#### Money and dates

- **Minor units** — the smallest unit of a currency. For CAD, cents; `1234` means `$12.34`.
- **Exponent** — how many decimal places a currency uses. CAD has exponent 2.
- **Decimal separator** — the dot or comma between the whole and fractional parts.
- **Locale** — a language/region setting that decides formatting, such as `en-CA`.
- **`Intl`** — JavaScript's built-in international formatting tools, such as `Intl.NumberFormat`.
- **UUID** — a long random id, such as `03408428-3ca2-...`, used to identify a row.
- **ISO date** — a date written `YYYY-MM-DD`; an ISO instant also includes time and `Z`.
- **Local date** — the date on the user's own clock, not UTC.
- **Month key** — a month written `YYYY-MM`.
- **Leap year** — a year with 366 days; February has 29 days.

#### Storage

- **IndexedDB** — the browser's built-in database for larger amounts of local data.
- **Dexie** — a small library that makes IndexedDB easier to use.
- **Table** — a collection of rows of one shape, such as `entries`.
- **Schema** — the description of tables, keys, and indexes.
- **Primary key** — the field that uniquely identifies a row.
- **Index** — a lookup structure that makes queries by a field fast.
- **Transaction** — a group of database writes that all succeed or all fail.
- **Live query** — a query that re-runs automatically when the data changes.
- **`bulkGet`** — read several rows by id in one call.
- **`startsWith` query** — find rows whose text starts with a prefix; used for `2026-11`.
- **Seed** — initial data inserted on first run.
- **Idempotent** — safe to run more than once with the same result.

#### UI and browser

- **React** — a library for building screens from components.
- **Component** — a function that takes props and returns a description of HTML.
- **Props** — the inputs a component receives.
- **State** — values a component remembers between renders.
- **Hook** — a React function that adds behavior, such as `useState` or `useEffect`.
- **`useState`** — remember a value and re-render when it changes.
- **`useEffect`** — run code after a render, often to subscribe or clean up.
- **`useRef`** — keep a value across renders without causing a re-render.
- **`useLiveQuery`** — a Dexie hook that re-runs a query when the database changes.
- **Render** — turn a component into HTML on screen.
- **JSX / TSX** — HTML-like syntax inside JavaScript/TypeScript.
- **Event** — something the user does, such as a click or a form submit.
- **`preventDefault`** — stop the browser's default action, such as a page reload on form submit.
- **Form** — the HTML element that groups inputs and a submit button.
- **DOM** — the browser's tree of page elements.
- **Accessibility (a11y)** — making the app usable with a screen reader or keyboard.
- **`aria-pressed`** — tells a screen reader whether a toggle button is selected.
- **`aria-describedby`** — points an input at the text that describes its error.
- **`fieldset` / `legend`** — group related inputs and give the group a name.
- **`htmlFor`** — connects a label to an input by id.
- **Hash route** — the part of the URL after `#`, used to choose a screen without a server.
- **Query string** — the `?m=2026-11&c=cat_fun` part of a URL.
- **`URLSearchParams`** — a helper for reading and writing query strings.
- **`encodeURIComponent`** — turn text into a safe URL piece.
- **`navigate`** — change the current URL hash.
- **History replace** — replace the current history entry so Back does not return to a stale page.

#### Build, PWA, and testing

- **PWA** — a web app that can be installed and work offline.
- **Service worker** — a background script that caches files for offline use.
- **Precache** — download and store files before they are needed.
- **Persistent storage** — a browser promise to keep data even when storage is tight.
- **Vite** — the build tool that serves and bundles the app.
- **Vitest** — the test runner used by the project.
- **React Testing Library** — helpers for testing React components by their visible behavior.
- **`fake-indexeddb`** — an in-memory IndexedDB used by tests and this notebook.
- **Test seam** — a place where tests can enter the code without mocking everything.

#### Notebook and runtime

- **Jupyter** — the notebook environment that runs cells.
- **Kernel** — the process that executes a notebook's cells. Here it is Deno.
- **Deno** — a TypeScript/JavaScript runtime that can run TypeScript directly.
- **`deno.json`** — Deno's configuration file, including the import map.
- **Import map** — a table that maps short names like `"dexie"` to real package URLs.
- **Sloppy imports** — a Deno flag that lets `./calendar` mean `./calendar.ts`.
- **`nbconvert`** — a command that runs a notebook from the terminal.
- **`jsdom`** — a fake browser DOM for running React outside a browser.
- **`act`** — a React test helper that waits for updates to finish.
- **`renderToStaticMarkup`** — turn a React element into an HTML string without a browser.

#### JavaScript built-ins and syntax you saw

- **`console.log`** — print a value to the notebook output.
- **`JSON.stringify`** — turn a value into text so you can compare or print it.
- **`Array.from`** — turn an array-like value (such as a list of DOM nodes) into a real array.
- **`.map`** — make a new array by transforming each item.
- **`.filter`** — make a new array with only the items that pass a test.
- **`.reduce`** — combine all items into one value, such as a sum.
- **`.sort`** — reorder an array using a comparison function.
- **`.find`** — return the first item that passes a test.
- **`.some`** — return `true` if at least one item passes a test.
- **`.includes`** — return `true` if a string or array contains a value.
- **`.startsWith`** — return `true` if a string begins with a prefix.
- **`.slice`** — copy part of a string or array.
- **`.padStart` / `.padEnd`** — add characters to the start or end until a length is reached.
- **`.localeCompare`** — compare two strings in a locale-aware order; returns a negative, zero, or positive number.
- **`Object.keys`** — list an object's field names.
- **`instanceof`** — check whether a value was created by a particular class.
- **`try` / `catch` / `finally`** — run code that might fail, handle the error, and optionally run cleanup.
- **`void`** — explicitly ignore a returned promise or value.
- **Non-null assertion (`!`)** — tell TypeScript "this value is not `null` or `undefined`". It does not change the runtime value.
- **`as any`** — tell TypeScript to stop checking a value's type. Use sparingly; it can hide bugs.
- **`React.createElement`** — the function JSX compiles to; it builds a React element from a component and props.
- **`createRoot`** — React's entry point for rendering a component tree into a DOM element.
- **`StrictMode`** — a React wrapper that runs extra checks in development; it does not render visible HTML.
- **`FormEvent`** — the TypeScript type for a form submit event.
- **`preventDefault`** — stop the browser's default action for an event.
- **`window.location`** — the browser object that holds the current URL.
- **`hashchange`** — the browser event fired when the part after `#` changes.
- **`decodeURIComponent`** — turn URL-encoded text back into normal text.
- **`navigator.storage.persist`** — the browser API that asks for persistent storage.
- **`inputMode`** — a hint to the phone keyboard about which keys to show.
- **`autoComplete`** — a hint to the browser about filling a field automatically.
- **`maxLength`** — the maximum number of characters an input accepts.
- **`disabled`** — prevent a button or input from being used.
- **`role`** — tells assistive technology what kind of element this is, such as `alert`.
- **`aria-label`** — a text label for screen readers when there is no visible label.
- **`aria-describedby`** — points an input at the element that describes it, often an error.
- **`aria-pressed`** — tells a screen reader whether a toggle button is on.
- **`htmlFor`** — connects a `<label>` to an input by id.
- **`fieldset` / `legend`** — group related inputs and give the group a name.
- **`Math.abs`** — absolute value, dropping the minus sign.
- **`Math.floor`** — round down to a whole number.
- **`Number.isInteger`** — check for a whole number.
- **`Number.isSafeInteger`** — check for a whole number JavaScript can represent exactly.
- **`Date`** — JavaScript's date/time object.
- **`setTimeout`** — run a function after a delay; used by `sleep`.
- **`Promise`** — a value that will exist later; `await` waits for it.
- **`virtual:pwa-register`** — a build-time module supplied by `vite-plugin-pwa`; it is not a real file in `src/`.

### Quiz (⏱️ ~5 min)

Answer these five questions in your own words before opening the answers.

1. What does `parseAmount("12,34", 2)` return, and why is it not `12.34`?
2. Why does the app store `1234` instead of `12.34` for a `$12.34` expense?
3. What is the difference between `monthSummary("2026-11")` and `categoryEntries("2026-11", "cat_groceries")`?
4. Why does `addExpense` use a database transaction for the entry and the settings update?
5. You tap a category row on the Home screen. List the functions and modules that run, in order, until the category list appears.

## Solutions (hidden)

Open only after you have tried the exercise. Each block shows one way to fix the bug; your fix may look different and still be correct.

<details>
<summary><strong>Lab 5.1 — fix `daysInMonthBroken`</strong></summary>

```ts
function daysInMonthFixed(year: number, month: number): number {
  if (month === 2) {
    return isLeapYear(year) ? 29 : 28;
  }
  if (month === 4 || month === 6 || month === 9 || month === 11) {
    return 30;
  }
  return 31;
}
```

`isLeapYear` is already defined in the lab cell. The bug was the hard-coded `28`.
</details>

<details>
<summary><strong>Lab 5.2 — fix `isValidMonthKeyBroken`</strong></summary>

```ts
function isValidMonthKeyFixed(value: string): boolean {
  var match = /^(\d{4})-(0[1-9]|1[0-2])$/.exec(value);
  if (!match) {
    return false;
  }
  var year = Number(match[1]);
  return year >= 1 && year <= 9999;
}
```

The month part of the regex is `0[1-9]|1[0-2]`, and the year range is checked separately.
</details>

<details>
<summary><strong>Lab 5.3 — fix `addMonthsBroken`</strong></summary>

```ts
function addMonthsFixed(month: string, delta: number): string {
  if (!isValidMonthKey(month)) {
    throw new RangeError("Invalid month key");
  }
  if (!Number.isInteger(delta)) {
    throw new RangeError("Month delta must be an integer");
  }
  var year = Number(month.slice(0, 4));
  var monthNumber = Number(month.slice(5, 7));
  var total = year * 12 + (monthNumber - 1) + delta;
  var nextYear = Math.floor(total / 12);
  var nextMonth = total - nextYear * 12 + 1;
  if (nextYear < 1 || nextYear > 9999) {
    throw new RangeError("Month is outside the representable range");
  }
  return `${String(nextYear).padStart(4, "0")}-${String(nextMonth).padStart(2, "0")}`;
}
```

The trick is to count months from year 0, not to add to the month number directly.
</details>

<details>
<summary><strong>Lab 5.4 — fix `parseAmountBroken`</strong></summary>

```ts
function parseAmountFixed(input: string, exponent: number): number | null {
  if (!Number.isInteger(exponent) || exponent < 0 || exponent > 6) {
    throw new RangeError("Currency exponent must be an integer in 0..6");
  }
  var trimmed = input.trim();
  if (exponent === 0) {
    if (!/^\d+$/.test(trimmed)) {
      return null;
    }
    var whole = Number(trimmed);
    return Number.isSafeInteger(whole) ? whole : null;
  }
  var match = /^(\d+)(?:[.,](\d+))?$/.exec(trimmed);
  if (!match) {
    return null;
  }
  var wholeText = match[1] ?? "";
  var fractionText = match[2] ?? "";
  if (fractionText.length > exponent) {
    return null;
  }
  var amount =
    Number(wholeText) * 10 ** exponent +
    Number(fractionText.padEnd(exponent, "0"));
  return Number.isSafeInteger(amount) ? amount : null;
}
```

The missing line was `if (fractionText.length > exponent) return null;`.
</details>

<details>
<summary><strong>Lab 5.5 — fix `formatAmountInputBroken`</strong></summary>

```ts
function formatAmountInputFixed(amountMinor: number, exponent: number): string {
  if (!Number.isInteger(exponent) || exponent < 0 || exponent > 6) {
    throw new RangeError("Currency exponent must be an integer in 0..6");
  }
  if (!Number.isSafeInteger(amountMinor)) {
    throw new RangeError("Amount must be a safe integer");
  }
  var digits = String(Math.abs(amountMinor)).padStart(exponent + 1, "0");
  var sign = amountMinor < 0 ? "-" : "";
  if (exponent === 0) {
    return `${sign}${digits}`;
  }
  var split = digits.length - exponent;
  return `${sign}${digits.slice(0, split)}.${digits.slice(split)}`;
}
```

It pads the digits first, then inserts the decimal point. That keeps `"45.00"` and `"0.05"`.
</details>

<details>
<summary><strong>Lab 5.6 — fix `requireAccountNameBroken`</strong></summary>

```ts
function requireAccountNameFixed(name: string): string {
  var trimmed = name.trim();
  if (trimmed.length === 0 || trimmed.length > 60) {
    throw new LedgerError(
      "invalid-name",
      "Enter an account name of 60 characters or fewer.",
    );
  }
  return trimmed;
}
```

`trim()` removes the spaces; the length check rejects empty and over-60 names.
</details>

<details>
<summary><strong>Lab 5.7 — fix `monthSummaryBroken`</strong></summary>

```ts
async function monthSummaryFixed(db: BudgetDatabase, month: string) {
  var entries = await db.entries.where("date").startsWith(month).toArray();
  var expenses = entries.filter((entry) => entry.kind === "expense");
  var totalMinor = expenses.reduce((sum, entry) => sum + entry.amountMinor, 0);
  return { totalMinor };
}
```

Only `kind === "expense"` counts as spending. Income and transfers are excluded.
</details>

<details>
<summary><strong>Lab 5.8 — fix `categoryEntriesBroken`</strong></summary>

```ts
async function categoryEntriesFixed(
  db: BudgetDatabase,
  month: string,
  categoryId: string,
) {
  var entries = await db.entries.where("date").startsWith(month).toArray();
  var rows = entries
    .filter((entry) => entry.kind === "expense")
    .filter(
      (entry) =>
        (entry.categoryId ?? UNCATEGORIZED_CATEGORY_ID) === categoryId,
    )
    .sort(
      (left, right) =>
        right.date.localeCompare(left.date) ||
        right.id.localeCompare(left.id),
    );
  return {
    rows,
    totalMinor: rows.reduce((sum, row) => sum + row.amountMinor, 0),
  };
}
```

The sort compares `right` to `left`, which puts the newest date first. The id tie-breaker keeps the order stable.
</details>

<details>
<summary><strong>Lab 5.9 — fix `parseHashBroken`</strong></summary>

```ts
function parseHashFixed(hash: string, today: string): any {
  var withoutHash = hash.startsWith("#") ? hash.slice(1) : hash;
  var queryIndex = withoutHash.indexOf("?");
  var path =
    queryIndex === -1 ? withoutHash : withoutHash.slice(0, queryIndex);
  var query = queryIndex === -1 ? "" : withoutHash.slice(queryIndex + 1);

  if (path === "/add") {
    return { name: "add" };
  }
  if (path === "/accounts") {
    return { name: "accounts" };
  }

  var requestedMonth = new URLSearchParams(query).get("m");
  var month =
    requestedMonth !== null && isValidMonthKey(requestedMonth)
      ? requestedMonth
      : monthOf(today);

  if (path === "/category") {
    var categoryId = new URLSearchParams(query).get("c");
    if (categoryId === null || categoryId === "") {
      return { name: "home", month };
    }
    return { name: "category", month, categoryId };
  }

  if (path.startsWith("/entry/")) {
    var raw = path.slice("/entry/".length);
    if (raw === "" || raw.includes("/")) {
      return { name: "home", month: monthOf(today) };
    }
    try {
      var entryId = decodeURIComponent(raw);
      if (entryId === "") {
        return { name: "home", month: monthOf(today) };
      }
      return { name: "entry", entryId };
    } catch {
      return { name: "home", month: monthOf(today) };
    }
  }

  return { name: "home", month };
}
```

The path checks must come before the home fallback. The real `parseHash` in `src/app/router.ts` is the authority.
</details>

<details>
<summary><strong>Lab 5.10 — fix `accountLabelBroken`</strong></summary>

```ts
function accountLabelFixed(row: any, ambiguousNames: Set<string>): string {
  if (row.accountType !== null && ambiguousNames.has(row.accountName)) {
    return `${row.accountName} · ${ACCOUNT_TYPE_LABELS[row.accountType]}`;
  }
  return row.accountName;
}
```

Add the type only when the name is ambiguous and the type exists.
</details>

<details>
<summary><strong>Lab 5.11 — fix `duplicateAccountNamesBroken`</strong></summary>

```ts
function duplicateAccountNamesFixed(rows: any[]): Set<string> {
  var idsByName = new Map<string, Set<string>>();
  for (var row of rows) {
    if (row.accountType === null) {
      continue;
    }
    var ids = idsByName.get(row.accountName) ?? new Set<string>();
    ids.add(row.accountId);
    idsByName.set(row.accountName, ids);
  }
  return new Set(
    [...idsByName]
      .filter(([, ids]) => ids.size > 1)
      .map(([name]) => name),
  );
}
```

A name is ambiguous only when it has more than one **different** id. The broken copy counted names, not ids.
</details>

<details>
<summary><strong>Lab 5.13 — fix the trace failure</strong></summary>

The fix is to restore the settings row before adding:

```ts
await trace2Ledger.initialize();
var recovered = await trace2Ledger.addExpense({
  amountMinor: 1234,
  categoryId: "cat_groceries",
  accountId: trace2Account.id,
  date: "2026-11-04",
});
```

The deeper lesson: the transaction rolled back the entry when the settings read failed, so the database was not left with a half-saved expense.
</details>

<details>
<summary><strong>Quiz answers</strong></summary>

1. `parseAmount("12,34", 2)` returns `1234`. The comma is accepted as a decimal separator, and exponent 2 means the result is in cents.
2. The app stores `1234` because integers avoid floating-point rounding errors. `12.34` cannot be represented exactly in binary floating point; `1234` can.
3. `monthSummary("2026-11")` groups **all** expenses in the month by category and returns totals. `categoryEntries("2026-11", "cat_groceries")` returns the individual expense rows for **one** category in that month.
4. The entry write and the `lastAccountId` update must succeed or fail together. A transaction prevents a saved expense with stale settings, or a settings update with no expense.
5. Home button → `navigate({ name: "category", month, categoryId })` → `routeToHash` → `window.location.hash` → `hashchange` → `useHashRoute` → `parseHash` → `App` switches to `CategoryEntriesScreen` → `ledger.categoryEntries(month, categoryId)` → `db.entries.where("date").startsWith(month)` → filter by category → look up account names → render rows.
</details>